# 1-Minute OTC Strategies — Targeting 80%

**Goal:** Find a strategy that achieves 80%+ accuracy on true 1-minute trades (non-overlapping windows).

**Known facts:**
- OTC 5-second candle data, 122K candles, 8 days
- Follow-last on 1-min = 50% (random)
- Minute-level returns have zero autocorrelation
- BUT: live bot with v2.5 (skip tiny) shows ~62% on 65 trades
- Competitor confirmed 80% on 1-minute trades

**All backtests use corrected methodology:**
- True 1-minute windows (entry → entry + 60s)
- Non-overlapping predictions
- Full martingale simulation with bust tracking

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, random, warnings
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

# ═══════════════════════════════════════════════════════════════
# LOAD 1-SECOND DATA & BUILD 1-MINUTE TRADE DATA
# ═══════════════════════════════════════════════════════════════

# Use 1-second candle data ONLY (don't mix with 5s)
raw = pd.read_csv('../data/eurusd_otc_1s.csv')
raw['datetime'] = pd.to_datetime(raw['timestamp'], unit='s')
raw = raw.set_index('datetime').sort_index()
raw['second'] = raw.index.second

print(f"Raw 1-second candles: {len(raw):,}")
print(f"Date range: {raw.index[0]} → {raw.index[-1]}")
print(f"Hours of data: {len(raw) / 3600:.0f}")

# Build 1-minute trade data using 1-second precision
# Entry at :00, result at next :00 (60 seconds later)
c00 = raw[raw['second'] == 0][['open', 'high', 'low', 'close']].copy()

# Get prices at every second within the minute (1s resolution!)
for sec in range(1, 60):
    sec_data = raw[raw['second'] == sec][['close']].copy()
    sec_data.index = sec_data.index - pd.Timedelta(seconds=sec)  # Align to :00
    c00[f'c_{sec:02d}'] = sec_data['close']

# Result: next :00 close (shift(-1) on filtered = 1 minute ahead ✓)
c00['result_close'] = c00['close'].shift(-1)
c00['went_up'] = (c00['result_close'] > c00['close']).astype(int)
c00['move'] = c00['result_close'] - c00['close']
c00['move_abs'] = c00['move'].abs()
c00['prev_went_up'] = c00['went_up'].shift(1)
c00['hour'] = c00.index.hour
c00['dow'] = c00.index.dayofweek

trades = c00.dropna(subset=['went_up', 'prev_went_up']).copy()

follow_acc = (trades['went_up'] == trades['prev_went_up']).mean()
print(f"\nTrade dataset: {len(trades):,} minutes")
print(f"Candle resolution: 1 second (60 data points per minute)")
print(f"Follow-last accuracy (1-min, corrected): {follow_acc:.1%}")

# ── Quick sanity checks ──
# Verify timing
print(f"\nSanity checks:")
print(f"  :00 candles: {len(c00)}, spacing: {(c00.index[1]-c00.index[0]).total_seconds():.0f}s")
print(f"  Has c_01: {c00['c_01'].notna().sum()}, c_30: {c00['c_30'].notna().sum()}, c_59: {c00['c_59'].notna().sum()}")

# Test: follow last with 1s data
print(f"\n--- Follow-Last with 1s Precision ---")
print(f"  :00→:00 follow last: {follow_acc:.1%}")

# Entry at exact second 2, 3, 4 (simulating click delay)
for entry_sec in [1, 2, 3, 4, 5]:
    col = f'c_{entry_sec:02d}'
    valid = trades.dropna(subset=[col])
    entry_p = valid[col].values
    result_p = valid['result_close'].values
    trade_up = (result_p > entry_p).astype(int)
    
    # Follow previous minute's :00→:00 direction
    acc = (trade_up == valid['prev_went_up'].values).mean()
    print(f"  :{entry_sec:02d}→:00 follow last: {acc:.1%} ({len(valid)} trades)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CORRECTED BACKTEST ENGINE — Fixed Martingale
# ═══════════════════════════════════════════════════════════════

def backtest(predictions, actuals, base_stake=1.0, payout=0.85, max_losses=8, max_exposure=500):
    """
    Run CORRECT martingale backtest.
    Each level's win recovers ALL previous losses + base profit.
    """
    balance = 1000
    history = [balance]
    stake = base_stake
    cumulative_loss = 0.0  # Track total lost in current progression
    consec = 0
    wins = 0; losses = 0; busts = 0
    
    for pred, actual in zip(predictions, actuals):
        if consec >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            cumulative_loss = 0.0
            consec = 0
        if stake > balance:
            break
        
        if int(pred) == int(actual):
            balance += stake * payout
            wins += 1
            stake = base_stake
            cumulative_loss = 0.0
            consec = 0
        else:
            balance -= stake
            losses += 1
            consec += 1
            cumulative_loss += stake
            # CORRECT: next stake must recover ALL losses + earn base profit
            stake = round((cumulative_loss + base_stake * payout) / payout, 2)
        
        history.append(balance)
    
    total = wins + losses
    hours = len(predictions) / 60
    profit = balance - 1000
    return {
        'win_rate': wins / total * 100 if total > 0 else 0,
        'wins': wins, 'losses': losses,
        'trades': total,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
        'max_dd': 1000 - min(history),
        'history': history,
    }

def show_result(name, r):
    bust_str = f"✓ 0 busts" if r['busts'] == 0 else f"❌ {r['busts']} busts"
    profit_str = f"✓ +${r['profit']:.0f}" if r['profit'] > 0 else f"❌ -${abs(r['profit']):.0f}"
    print(f"  {name:<35} {r['win_rate']:>5.1f}% | {r['trades']:>5} trades | {bust_str:<12} | {profit_str}")

# Verify the fix
print("Martingale stake progression (FIXED):")
stake = 1.0
cum = 0.0
for level in range(1, 9):
    win_amount = stake * 0.85
    net_if_win = win_amount - cum
    print(f"  Level {level}: stake=${stake:.2f}, win=${win_amount:.2f}, net if win=${net_if_win:+.2f}")
    cum += stake
    stake = round((cum + 1.0 * 0.85) / 0.85, 2)
print(f"  Bust cost: ${cum:.2f}")
print(f"\n  Every win now recovers ALL losses + $0.85 profit ✓")

## Strategy 1: Intra-Minute Price Shape

We have 12 five-second candles within each minute. The SHAPE of the price path within the minute might predict the NEXT minute's direction. This is data the competitor could be using that our follow-last strategy ignores.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STRATEGY 1: Intra-Minute Features with 1-SECOND Resolution
# ═══════════════════════════════════════════════════════════════

print("STRATEGY 1: Intra-Minute Signals (1-second resolution)")
print("=" * 60)

t = trades.dropna(subset=[f'c_{s:02d}' for s in [5, 10, 15, 25, 30, 45, 55]]).copy()

# Build features from 1-second data
# Price at every 5 seconds (for comparable signals to before)
prices_5s = t[['close'] + [f'c_{s:02d}' for s in range(5, 60, 5)]].values
diffs_5s = np.diff(prices_5s, axis=1)

# Up ratio of 5s steps
t['up_ratio'] = (diffs_5s > 0).sum(axis=1) / 11

# Close position (where :59 close sits relative to minute's range)
t['close_position'] = 0.5  # default
if 'c_59' in t.columns:
    minute_prices = t[['close'] + [f'c_{s:02d}' for s in range(1, 60)]].dropna(axis=1).values
    minute_high = minute_prices.max(axis=1)
    minute_low = minute_prices.min(axis=1)
    minute_range = minute_high - minute_low
    last_price = t['c_59'].values if 'c_59' in t.columns else t['c_55'].values
    t['close_position'] = np.where(minute_range > 0, 
        (last_price - minute_low) / minute_range, 0.5)

# Acceleration/deceleration
t['first_half'] = t['c_25'] - t['close']
t['second_half'] = t['c_55'] - t['c_30']
t['accelerating'] = (t['second_half'].abs() > t['first_half'].abs()).astype(int)
t['reversed'] = ((t['first_half'] * t['second_half']) < 0).astype(int)

# ── Test CAUSAL signals only ──
# At :00 of minute N, you can only see data from minute N-1
print(f"\nDataset: {len(t):,} minutes (1s resolution)\n")
print("CAUSAL signals only (data from PREVIOUS minute):\n")

signals = []

# Baseline
acc = (t['went_up'] == t['prev_went_up']).mean()
signals.append(('Follow last :00→:00 result', acc, len(t)))

# Previous minute's last N seconds direction (CAUSAL - known at :00)
for last_n in [1, 2, 3, 5, 10, 15, 30]:
    start_sec = 60 - last_n
    start_col = f'c_{start_sec:02d}'
    end_col = 'c_59'
    if start_col in t.columns and end_col in t.columns:
        prev_sig = (t[end_col].shift(1) > t[start_col].shift(1)).astype(int)
        valid = t.dropna(subset=[start_col])
        acc = (valid['went_up'] == prev_sig.loc[valid.index]).dropna().mean()
        signals.append((f'Prev min last {last_n}s direction', acc, len(valid)))

# Previous minute's full direction :00→:59 (CAUSAL)
prev_full = (t['c_59'].shift(1) > t['close'].shift(1)).astype(int)
acc = (t['went_up'] == prev_full).dropna().mean()
signals.append(('Prev min full direction', acc, len(t)))

# Previous minute's close position (CAUSAL)
prev_close_pos = t['close_position'].shift(1)
high_pos = prev_close_pos > 0.7
low_pos = prev_close_pos < 0.3
if high_pos.sum() > 50:
    acc = t.loc[high_pos, 'went_up'].mean()
    signals.append(('Prev close near HIGH → UP', acc, high_pos.sum()))
if low_pos.sum() > 50:
    acc = 1 - t.loc[low_pos, 'went_up'].mean()
    signals.append(('Prev close near LOW → DOWN', acc, low_pos.sum()))

# First 1-3 seconds of CURRENT minute (CAUSAL if entering at :03-:05)
for first_n in [1, 2, 3]:
    col = f'c_{first_n:02d}'
    if col in t.columns:
        initial_up = (t[col] > t['close']).astype(int)
        # Trade: enter at :N, result at next :00
        result_p = t['result_close']
        trade_up = (result_p > t[col]).astype(int)
        acc = (trade_up == initial_up).mean()
        signals.append((f'First {first_n}s momentum (:{first_n:02d}→:00)', acc, len(t)))

# Fade versions of current minute start
for first_n in [1, 2, 3]:
    col = f'c_{first_n:02d}'
    if col in t.columns:
        initial_up = (t[col] > t['close']).astype(int)
        trade_up = (t['result_close'] > t[col]).astype(int)
        acc = (trade_up != initial_up).mean()
        signals.append((f'First {first_n}s FADE (:{first_n:02d}→:00)', acc, len(t)))

signals.sort(key=lambda x: x[1], reverse=True)

print(f"  {'Signal':<40} {'Accuracy':>8} {'Trades':>7}")
print(f"  {'-'*60}")
for name, acc, n in signals:
    marker = " ★" if acc > 0.54 else ""
    print(f"  {name:<40} {acc:>7.1%} {n:>7}{marker}")

# Chart
fig, ax = plt.subplots(figsize=(14, 7))
names = [s[0][:38] for s in signals]
accs = [s[1] for s in signals]
colors = ['lime' if a > 0.54 else ('gold' if a > 0.51 else 'red') for a in accs]
ax.barh(range(len(signals)), accs, color=colors, alpha=0.7)
ax.axvline(x=0.5, color='white', linestyle=':', alpha=0.3)
ax.axvline(x=0.5405, color='red', linestyle='--', label='Break-even (54%)')
ax.axvline(x=0.80, color='yellow', linestyle='--', label='Target (80%)')
ax.set_yticks(range(len(signals)))
ax.set_yticklabels(names, fontsize=8)
ax.set_title('1-Minute CAUSAL Signals — 1-Second Resolution')
ax.set_xlabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## Strategy 2: Machine Learning on All Features

Let Random Forest / XGBoost find patterns we can't see manually. Use all 12 intra-minute prices + lagged features.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STRATEGY 2: ML — CAUSAL FEATURES ONLY (no look-ahead)
# ═══════════════════════════════════════════════════════════════
# RULE: Only use data from BEFORE the trade is placed.
# If entering at :00, all features must come from the PREVIOUS minute.

print("STRATEGY 2: ML with CAUSAL Features Only (No Look-Ahead)")
print("=" * 60)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

t_ml = t.copy()

# All features LAGGED by 1 (previous minute's data)
price_cols = ['close'] + [f'c_{s:02d}' for s in range(5, 60, 5)]
prices_arr = t_ml[price_cols].values
steps = np.diff(prices_arr, axis=1)

# Previous minute's 11 step changes
for j in range(11):
    t_ml[f'prev_step_{j}'] = pd.Series(steps[:, j], index=t_ml.index).shift(1)

# Previous minute's summary features (ALL shifted by 1)
t_ml['prev_up_ratio'] = t_ml['up_ratio'].shift(1)
t_ml['prev_close_pos'] = t_ml['close_position'].shift(1)
t_ml['prev_move_abs'] = t_ml['move_abs'].shift(1)
t_ml['prev_direction'] = t_ml['prev_went_up']  # already lagged
t_ml['prev_accel'] = t_ml['accelerating'].shift(1)
t_ml['prev_reversed'] = t_ml['reversed'].shift(1)

# 2-minute-ago features
t_ml['prev2_direction'] = t_ml['went_up'].shift(2)
t_ml['prev2_up_ratio'] = t_ml['up_ratio'].shift(2)
t_ml['prev2_move_abs'] = t_ml['move_abs'].shift(2)

# Time features
t_ml['hour_feat'] = t_ml['hour']

t_ml = t_ml.dropna()

feature_names = [f'prev_step_{j}' for j in range(11)] + [
    'prev_up_ratio', 'prev_close_pos', 'prev_move_abs', 'prev_direction',
    'prev_accel', 'prev_reversed',
    'prev2_direction', 'prev2_up_ratio', 'prev2_move_abs',
    'hour_feat',
]

X = t_ml[feature_names].values.astype(float)
y = t_ml['went_up'].values

print(f"Features: {X.shape[1]} (ALL from previous minutes)")
print(f"Samples: {X.shape[0]}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n5-Fold Cross-Validated Accuracy:")
print(f"  (Break-even = 54%, Target = 80%)\n")

follow_acc = (t_ml['went_up'] == t_ml['prev_went_up']).mean()
print(f"  Follow last result:         {follow_acc:.1%}")

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X_scaled, y, cv=5, scoring='accuracy')
print(f"  Random Forest:              {rf_scores.mean():.1%} (±{rf_scores.std():.1%})")

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
gb_scores = cross_val_score(gb, X_scaled, y, cv=5, scoring='accuracy')
print(f"  Gradient Boosting:          {gb_scores.mean():.1%} (±{gb_scores.std():.1%})")

# Feature importance
rf.fit(X_scaled, y)
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Feature Importances:")
for _, row in importances.head(10).iterrows():
    bar = '█' * int(row['importance'] * 200)
    print(f"  {row['feature']:<25} {row['importance']:.3f} {bar}")

# Out-of-sample
split = int(len(X) * 0.7)
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y[:split], y[split:]

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_oos = (rf_pred == y_test).mean()

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_oos = (gb_pred == y_test).mean()

follow_oos = (t_ml.iloc[split:]['prev_went_up'].values == y_test).mean()

print(f"\nOut-of-Sample (last 30%):")
print(f"  Follow last:     {follow_oos:.1%}")
print(f"  Random Forest:   {rf_oos:.1%}")
print(f"  Gradient Boost:  {gb_oos:.1%}")

# Backtest
print(f"\nFull Martingale Backtest (out-of-sample):")
r_follow = backtest(t_ml.iloc[split:]['prev_went_up'].values, y_test)
show_result("Follow last result", r_follow)

r_rf_oos = backtest(rf_pred, y_test)
show_result("Random Forest (causal)", r_rf_oos)

r_gb_oos = backtest(gb_pred, y_test)
show_result("Gradient Boost (causal)", r_gb_oos)

## Strategy 3: Entry Timing — First 5 Seconds as Signal

Instead of entering blind at :00, wait a few seconds and use the initial price movement as a signal. If price moves UP in the first 5 seconds, bet it continues UP for the remaining 55 seconds.

## Strategy 3: Simulate Exact Live Bot Behavior (v2, v2.5, v3, v4)

Simulate the ACTUAL bot flow minute-by-minute:
1. At :00, check if we should trade (skip filter)
2. If SKIP: observe price, update direction, set entry for next observation, clear result price
3. If TRADE: place trade, wait for result at next :00
4. Result updates direction for next trade

This matches `run.py` lines 218-244 exactly.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SIMULATE EXACT LIVE BOT BEHAVIOR — v2, v2.5, v3, v4
# ═══════════════════════════════════════════════════════════════
# Replicate run.py logic exactly, minute by minute.

print("SIMULATING EXACT LIVE BOT BEHAVIOR")
print("=" * 60)

def simulate_bot(df_minutes, min_move=0.0, min_age=0, base_stake=1.0, 
                  payout=0.85, max_losses=8, max_exposure=200, label=""):
    """
    Simulate the exact v2/v2.5/v3/v4 bot logic on minute-by-minute data.
    
    df_minutes: DataFrame with columns 'close', 'result_close', and optionally c_XX columns
                Index is :00 timestamps, one row per minute.
    
    The bot flow each minute:
    1. Check if should_trade (min_move filter on last observed move)
    2. If SKIP: observe current price, update direction, set new entry price, clear result price
    3. If TRADE: bet in last_direction, get result, update direction from result
    """
    balance = 1000
    history = [balance]
    stake = base_stake
    consec_losses = 0
    wins = 0
    losses = 0
    busts = 0
    skips = 0
    
    # Bot state (mirrors strategy class)
    last_result_direction = None  # "UP" or "DOWN"
    last_entry_price = None
    last_result_price = None
    momentum_age = 0
    
    results_log = []  # For analysis
    
    for i in range(len(df_minutes)):
        row = df_minutes.iloc[i]
        entry_price_now = row['close']       # Price at :00 (entry moment)
        result_price_now = row['result_close'] # Price at next :00 (result)
        
        if pd.isna(result_price_now):
            continue
        
        # ── Should we trade? ──
        should_trade = True
        
        # First trade: always enter
        if last_entry_price is None:
            should_trade = True
        else:
            # Min move filter (v2.5+)
            if min_move > 0 and last_result_price is not None:
                last_move = abs(last_result_price - last_entry_price)
                if last_move < min_move:
                    should_trade = False
            
            # When result price is None (cleared after skip), always trade
            if min_move > 0 and last_result_price is None:
                should_trade = True
            
            # Min age filter (v3)
            if min_age > 0 and momentum_age < min_age:
                should_trade = False
        
        if not should_trade:
            # ── SKIP: observe and update ──
            # Update direction from observed movement (run.py line 231-233)
            if last_entry_price is not None:
                observed_up = entry_price_now > last_entry_price
                last_result_direction = "UP" if observed_up else "DOWN"
            
            # Update momentum age
            # (simplified — track consecutive same direction)
            
            # Set entry for next observation (run.py line 237)
            last_entry_price = entry_price_now
            # Clear result price so next should_trade sees None (run.py line 239)
            last_result_price = None
            
            skips += 1
            results_log.append({'type': 'skip', 'minute': i})
            continue
        
        # ── TRADE ──
        # Direction
        if last_result_direction is None:
            # First trade — random
            direction = "UP" if random.random() > 0.5 else "DOWN"
        else:
            direction = last_result_direction
        
        # Martingale safety
        if consec_losses >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            consec_losses = 0
        
        if stake > balance:
            break
        
        # Place trade: entry at :00, result at next :00
        actual_up = result_price_now > entry_price_now
        bet_up = direction == "UP"
        won = bet_up == actual_up
        
        if won:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consec_losses = 0
            # Direction stays the same (we were right)
            last_result_direction = direction
        else:
            balance -= stake
            losses += 1
            consec_losses += 1
            stake = round((stake + base_stake) / payout, 2)
            # Direction flips (we were wrong, follow what actually happened)
            last_result_direction = "UP" if actual_up else "DOWN"
        
        # Update entry/result prices for next trade's skip filter
        last_entry_price = entry_price_now
        last_result_price = result_price_now
        
        # Update momentum age
        if i > 0:
            prev_up = df_minutes.iloc[i-1]['result_close'] > df_minutes.iloc[i-1]['close']
            if actual_up == prev_up:
                momentum_age += 1
            else:
                momentum_age = 1
        
        results_log.append({
            'type': 'trade', 'minute': i, 'won': won,
            'direction': direction, 'actual_up': actual_up,
            'stake': stake, 'balance': balance,
        })
        history.append(balance)
    
    total = wins + losses
    opportunities = total + skips
    hours = len(df_minutes) / 60
    profit = balance - 1000
    skip_pct = skips / opportunities * 100 if opportunities > 0 else 0
    
    return {
        'label': label,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'wins': wins, 'losses': losses,
        'trades': total,
        'skips': skips,
        'skip_pct': skip_pct,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
        'max_dd': 1000 - min(history) if history else 0,
        'max_consec_loss': max((len(list(g)) for k, g in __import__('itertools').groupby(
            [r['won'] for r in results_log if r['type'] == 'trade']) if not k), default=0),
        'history': history,
    }

# Prepare minute data
df_min = trades[['close', 'result_close']].copy()

# ── Run all strategy variants ──
strategies = []

# v2: no filter
r = simulate_bot(df_min, min_move=0, label="v2: No filter")
strategies.append(r)

# v2.5: skip tiny moves (< 0.00019)
r = simulate_bot(df_min, min_move=0.00019, label="v2.5: Skip tiny (0.00019)")
strategies.append(r)

# Test different min_move thresholds
for mm in [0.00005, 0.00010, 0.00015, 0.00019, 0.00025, 0.00030, 0.00040, 0.00058]:
    r = simulate_bot(df_min, min_move=mm, label=f"Skip >{mm:.5f}")
    strategies.append(r)

# v3: skip tiny + fresh reversals (age >= 2)
r = simulate_bot(df_min, min_move=0.00019, min_age=2, label="v3: Tiny + age>=2")
strategies.append(r)

# v4: large moves only
r = simulate_bot(df_min, min_move=0.00058, label="v4: Large only (0.00058)")
strategies.append(r)

# Sort by win rate
strategies.sort(key=lambda x: x['win_rate'], reverse=True)

print(f"\n{'Strategy':<28} {'Win%':>6} {'Trades':>6} {'Skip%':>6} {'MaxL':>5} {'Busts':>6} {'$/hr':>7} {'Profit':>9}")
print(f"{'-'*85}")
for s in strategies:
    bust_str = f"✓ 0" if s['busts'] == 0 else f"❌ {s['busts']}"
    profit_str = f"✓ +${s['profit']:.0f}" if s['profit'] > 0 else f"❌ ${s['profit']:.0f}"
    marker = " ★" if s['win_rate'] > 54 else ""
    print(f"{s['label']:<28} {s['win_rate']:>5.1f}% {s['trades']:>6} {s['skip_pct']:>5.1f}% "
          f"{s['max_consec_loss']:>5} {bust_str:>6} ${s['per_hour']:>6.2f} {profit_str:>9}{marker}")

# Plot equity curves for key strategies
fig, ax = plt.subplots(figsize=(16, 7))
for s in strategies:
    if s['label'] in ['v2: No filter', 'v2.5: Skip tiny (0.00019)', 
                       'v3: Tiny + age>=2', 'v4: Large only (0.00058)',
                       'Skip >0.00010', 'Skip >0.00030']:
        lw = 2 if 'v2.5' in s['label'] else 1
        ax.plot(s['history'], label=f"{s['label']} ({s['win_rate']:.1f}%)", linewidth=lw)

ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Exact Bot Simulation — Minute by Minute with Skip Logic')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Summary
print(f"\nKEY QUESTION: Does skip-and-observe create edge?")
v2 = [s for s in strategies if s['label'] == 'v2: No filter'][0]
v25 = [s for s in strategies if s['label'] == 'v2.5: Skip tiny (0.00019)'][0]
print(f"  v2 (no skip):  {v2['win_rate']:.1f}% on {v2['trades']} trades")
print(f"  v2.5 (skip):   {v25['win_rate']:.1f}% on {v25['trades']} trades, {v25['skip_pct']:.0f}% skipped")
print(f"  Difference:    {v25['win_rate'] - v2['win_rate']:+.1f}%")
if v25['win_rate'] > v2['win_rate'] + 2:
    print(f"  → YES! Skip logic adds {v25['win_rate'] - v2['win_rate']:.1f}% edge")
else:
    print(f"  → No significant difference from skip logic")

## Strategy 5: Deeper Exploration — What We Haven't Tried

We've been stuck on "predict direction from previous direction." But there are patterns beyond that:
1. **Open vs Close pricing** — IQ Option might use different exact prices than our candle close
2. **Price relative to a level** — mean reversion, round-trip patterns
3. **Volatility clustering** — size of moves is autocorrelated even if direction isn't
4. **Conditional patterns** — signal only exists in specific conditions
5. **Multi-lag non-linear patterns** — XOR-like relationships between lags
6. **Sub-minute structure** — how price moves WITHIN the minute

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5A: OPEN vs CLOSE — Does the price reference point matter?
# ═══════════════════════════════════════════════════════════════
# IQ Option might use OPEN of :00 candle (exact :00.000 price) 
# not CLOSE (price at :00.999). Let's test all combinations.

print("5A: Price Reference Points — Open vs Close")
print("=" * 60)

c = raw[raw['second'] == 0][['open', 'close']].copy()

# Also get the :59 candle's close (last price before :00 boundary)
c59 = raw[raw['second'] == 59][['close']].copy()
c59.index = c59.index + pd.Timedelta(seconds=1)  # Align to :00
c['prev_59_close'] = c59['close']

c = c.dropna()

# Test every combination of entry/result price reference
combos = []
for entry_ref, entry_col in [('open', 'open'), ('close', 'close'), ('prev_:59_close', 'prev_59_close')]:
    for result_ref, result_shift in [('next_open', 'open'), ('next_close', 'close')]:
        # Result is next minute's price
        result_prices = c[result_shift].shift(-1)
        entry_prices = c[entry_col]
        
        went_up = (result_prices > entry_prices).astype(int)
        prev_went_up = went_up.shift(1)
        valid = pd.DataFrame({'went_up': went_up, 'prev': prev_went_up}).dropna()
        
        acc = (valid['went_up'] == valid['prev']).mean()
        combos.append((f'{entry_ref} → {result_ref}', acc, len(valid)))

combos.sort(key=lambda x: x[1], reverse=True)
print(f"\n  {'Entry → Result':<35} {'Follow-Last':>10} {'Trades':>7}")
print(f"  {'-'*55}")
for name, acc, n in combos:
    marker = " ★" if acc > 0.52 else ""
    print(f"  {name:<35} {acc:>9.1%} {n:>7}{marker}")

# ═══════════════════════════════════════════════════════════════
# 5B: Mean Reversion — Does price tend to return to a level?
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("5B: Mean Reversion Patterns")
print(f"{'='*60}\n")

c_mr = trades.copy()

# Moving averages of the :00 close price
for window in [5, 10, 20, 50]:
    c_mr[f'ma_{window}'] = c_mr['close'].rolling(window).mean()

c_mr = c_mr.dropna()

# If price is ABOVE its moving average, does it tend to go DOWN (revert)?
print("  If price above MA → next minute goes DOWN? (mean reversion)")
for window in [5, 10, 20, 50]:
    above_ma = c_mr['close'] > c_mr[f'ma_{window}']
    below_ma = ~above_ma
    
    # When above MA: does price go DOWN more often?
    down_when_above = (c_mr.loc[above_ma, 'went_up'] == 0).mean()
    up_when_below = (c_mr.loc[below_ma, 'went_up'] == 1).mean()
    
    # Strategy: fade (go opposite of position relative to MA)
    fade_pred = (~above_ma).astype(int)  # Above MA → bet DOWN, Below → bet UP
    acc = (c_mr['went_up'] == fade_pred).mean()
    
    marker = " ★" if acc > 0.52 else ""
    print(f"  MA-{window:>2}: Above→Down={down_when_above:.1%}  Below→Up={up_when_below:.1%}  "
          f"Fade accuracy={acc:.1%}{marker}")

# ═══════════════════════════════════════════════════════════════
# 5C: Volatility Clustering & Conditional Patterns
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("5C: Volatility Clustering & Conditional Patterns")
print(f"{'='*60}\n")

c_vol = trades.copy()
c_vol['prev_move_abs'] = c_vol['move_abs'].shift(1)
c_vol['prev2_move_abs'] = c_vol['move_abs'].shift(2)
c_vol = c_vol.dropna()

# Is move SIZE autocorrelated? (even if direction isn't)
corr_size, pval = pearsonr(c_vol['move_abs'].values[1:], c_vol['prev_move_abs'].values[1:])
print(f"  Move SIZE autocorrelation: r={corr_size:.4f} (p={pval:.4f})")
print(f"  → {'Volatility clusters!' if pval < 0.01 else 'No clustering'}")

# If volatility clusters, does it help prediction?
# After a BIG move, is the next direction more predictable?
c_vol['prev_big'] = (c_vol['prev_move_abs'] > c_vol['prev_move_abs'].quantile(0.75)).astype(int)
c_vol['prev_small'] = (c_vol['prev_move_abs'] < c_vol['prev_move_abs'].quantile(0.25)).astype(int)

print(f"\n  Follow-last accuracy conditioned on previous move size:")
for label, mask in [('After BIG move (top 25%)', c_vol['prev_big'] == 1),
                     ('After SMALL move (bottom 25%)', c_vol['prev_small'] == 1),
                     ('After MEDIUM move (middle 50%)', (c_vol['prev_big'] == 0) & (c_vol['prev_small'] == 0))]:
    subset = c_vol[mask]
    acc = (subset['went_up'] == subset['prev_went_up']).mean()
    n = len(subset)
    marker = " ★" if acc > 0.52 else ""
    print(f"    {label:<35}: {acc:.1%} ({n} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# 5D: Non-linear Patterns — XOR, Alternating, Streaks
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("5D: Non-Linear Lag Patterns")
print(f"{'='*60}\n")

c_nl = trades.copy()
c_nl['lag1'] = c_nl['went_up'].shift(1)
c_nl['lag2'] = c_nl['went_up'].shift(2)
c_nl['lag3'] = c_nl['went_up'].shift(3)
c_nl['lag4'] = c_nl['went_up'].shift(4)
c_nl['lag5'] = c_nl['went_up'].shift(5)
c_nl = c_nl.dropna()

# XOR patterns: if lag1 XOR lag2 == 1, predict...?
c_nl['xor_12'] = (c_nl['lag1'] != c_nl['lag2']).astype(int)
for xor_val, label in [(1, 'Lag1 ≠ Lag2 (alternated)'), (0, 'Lag1 = Lag2 (continued)')]:
    mask = c_nl['xor_12'] == xor_val
    up_pct = c_nl.loc[mask, 'went_up'].mean()
    # If there's a bias toward UP or DOWN in this condition
    best_acc = max(up_pct, 1 - up_pct)
    best_dir = "UP" if up_pct > 0.5 else "DOWN"
    follow_acc = (c_nl.loc[mask, 'went_up'] == c_nl.loc[mask, 'lag1']).mean()
    print(f"  {label}: P(UP)={up_pct:.1%}, Follow={follow_acc:.1%}, Best={best_dir}({best_acc:.1%}) [{mask.sum()} trades]")

# Streak patterns: after N consecutive same direction, what happens?
print(f"\n  After N consecutive same-direction minutes:")
for streak_len in [2, 3, 4, 5, 6, 7, 8]:
    mask = True
    for lag in range(1, streak_len + 1):
        mask = mask & (c_nl[f'lag{min(lag, 5)}'] == c_nl['lag1'])
    if isinstance(mask, bool):
        continue
    subset = c_nl[mask]
    if len(subset) > 20:
        follow_acc = (subset['went_up'] == subset['lag1']).mean()
        fade_acc = 1 - follow_acc
        best = max(follow_acc, fade_acc)
        best_name = "Follow" if follow_acc > fade_acc else "Fade"
        marker = " ★" if best > 0.54 else ""
        print(f"    After {streak_len} same: {best_name}={best:.1%} ({len(subset)} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# 5E: Sub-Minute Structure — Price Path Shape
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("5E: Sub-Minute Price Path Analysis (1s resolution)")
print(f"{'='*60}\n")

# For each minute, compute price path features
t5 = trades.dropna(subset=[f'c_{s:02d}' for s in [10, 20, 30, 40, 50, 59]]).copy()

# Did price cross the entry level during the previous minute?
# (number of times price crossed the :00 open price)
prev_prices = []
for sec in range(1, 60):
    col = f'c_{sec:02d}'
    if col in t5.columns:
        prev_prices.append(col)

if len(prev_prices) > 10:
    price_matrix = t5[['close'] + prev_prices].values
    # Count zero crossings relative to the entry price
    relative = price_matrix - price_matrix[:, 0:1]  # Relative to :00
    signs = np.sign(relative)
    crossings = np.abs(np.diff(signs, axis=1))
    t5['num_crossings'] = (crossings > 0).sum(axis=1)
    
    # Shift for previous minute's crossings
    t5['prev_crossings'] = t5['num_crossings'].shift(1)
    t5_valid = t5.dropna(subset=['prev_crossings'])
    
    # Does previous minute's choppiness predict this minute?
    for label, mask in [('Prev smooth (≤2 crossings)', t5_valid['prev_crossings'] <= 2),
                         ('Prev choppy (5-10 crossings)', (t5_valid['prev_crossings'] >= 5) & (t5_valid['prev_crossings'] <= 10)),
                         ('Prev very choppy (>10)', t5_valid['prev_crossings'] > 10)]:
        subset = t5_valid[mask]
        if len(subset) > 50:
            follow_acc = (subset['went_up'] == subset['prev_went_up']).mean()
            marker = " ★" if follow_acc > 0.52 or follow_acc < 0.48 else ""
            print(f"  {label:<35}: follow={follow_acc:.1%} ({len(subset)} trades){marker}")

# Time spent above vs below entry during previous minute
if len(prev_prices) > 10:
    time_above = (relative[:, 1:] > 0).sum(axis=1) / relative[:, 1:].shape[1]
    t5['time_above_entry'] = time_above
    t5['prev_time_above'] = t5['time_above_entry'].shift(1)
    t5_v2 = t5.dropna(subset=['prev_time_above'])
    
    print(f"\n  Previous minute time spent above entry:")
    for label, lo, hi in [('Mostly below (<30%)', 0, 0.3), 
                           ('Mixed (30-70%)', 0.3, 0.7),
                           ('Mostly above (>70%)', 0.7, 1.01)]:
        mask = (t5_v2['prev_time_above'] >= lo) & (t5_v2['prev_time_above'] < hi)
        subset = t5_v2[mask]
        if len(subset) > 50:
            up_pct = subset['went_up'].mean()
            follow_acc = (subset['went_up'] == subset['prev_went_up']).mean()
            marker = " ★" if abs(up_pct - 0.5) > 0.02 else ""
            print(f"    {label:<25}: P(UP)={up_pct:.1%}, Follow={follow_acc:.1%} ({len(subset)}){marker}")

# ═══════════════════════════════════════════════════════════════
# 5F: Hour-of-Day Deep Dive
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("5F: Hour-of-Day — Does Follow-Last Work at Specific Hours?")
print(f"{'='*60}\n")

c_hour = trades.copy()
by_hour = c_hour.groupby('hour').apply(
    lambda g: pd.Series({
        'follow_acc': (g['went_up'] == g['prev_went_up']).mean(),
        'up_bias': g['went_up'].mean(),
        'count': len(g),
    })
).reset_index()

print(f"  {'Hour':>4} {'Follow%':>8} {'UP bias':>8} {'Trades':>7} {'Strategy':>20}")
print(f"  {'-'*55}")
for _, row in by_hour.iterrows():
    best_fixed = max(row['up_bias'], 1 - row['up_bias'])
    best_fixed_dir = "Always UP" if row['up_bias'] > 0.5 else "Always DOWN"
    best = max(row['follow_acc'], best_fixed)
    strategy = "Follow" if row['follow_acc'] >= best_fixed else best_fixed_dir
    marker = " ★" if best > 0.54 else ""
    print(f"  {int(row['hour']):>4} {row['follow_acc']:>7.1%} {row['up_bias']:>7.1%} {int(row['count']):>7} {strategy:>20}{marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6A: DIG INTO STREAKS — The one signal that showed life
# ═══════════════════════════════════════════════════════════════

print("6A: Streak Analysis — Deep Dive")
print("=" * 60)

c_s = trades.copy()
c_s['dir'] = c_s['went_up']

# Compute streak length at each point
streaks = []
count = 1
prev = None
for val in c_s['dir'].values:
    if val == prev:
        count += 1
    else:
        count = 1
    streaks.append(count)
    prev = val

c_s['streak'] = streaks
c_s['prev_streak'] = c_s['streak'].shift(1)
c_s['prev_dir'] = c_s['dir'].shift(1)
c_s = c_s.dropna()

# After a streak of N, what happens?
print(f"\n  After a streak of N same-direction minutes:")
print(f"  {'Streak':>7} {'P(continue)':>12} {'Follow':>8} {'Fade':>8} {'Best':>8} {'Trades':>7}")
print(f"  {'-'*60}")

for streak_len in range(1, 15):
    mask = c_s['prev_streak'] == streak_len
    subset = c_s[mask]
    if len(subset) < 20:
        continue
    continues = (subset['dir'] == subset['prev_dir']).mean()
    follow = continues
    fade = 1 - continues
    best = max(follow, fade)
    best_name = "Follow" if follow >= fade else "Fade"
    marker = " ★" if best > 0.54 else ""
    print(f"  {streak_len:>7} {continues:>11.1%} {follow:>7.1%} {fade:>7.1%} {best_name:>5}={best:.1%} {len(subset):>7}{marker}")

# Visualize
streak_data = []
for streak_len in range(1, 15):
    mask = c_s['prev_streak'] == streak_len
    subset = c_s[mask]
    if len(subset) >= 20:
        continues = (subset['dir'] == subset['prev_dir']).mean()
        streak_data.append({'streak': streak_len, 'continue_pct': continues, 'count': len(subset)})

sdf = pd.DataFrame(streak_data)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['lime' if p > 0.54 else ('gold' if p > 0.50 else 'red') for p in sdf['continue_pct']]
axes[0].bar(sdf['streak'], sdf['continue_pct'], color=colors, alpha=0.7)
axes[0].axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
axes[0].axhline(y=0.5405, color='red', linestyle='--', label='Break-even')
axes[0].set_title('P(streak continues) by streak length')
axes[0].set_xlabel('Current streak length')
axes[0].set_ylabel('P(continues)')
axes[0].legend()
axes[1].bar(sdf['streak'], sdf['count'], color='cyan', alpha=0.5)
axes[1].set_title('Sample count by streak length')
axes[1].set_xlabel('Current streak length')
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 6B: HOURLY BIAS — Stable across days?
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("6B: Hourly UP/DOWN Bias — Stable Across Days?")
print(f"{'='*60}\n")

c_h = trades.copy()
c_h['date'] = c_h.index.date

print("  Is hour 21's DOWN bias consistent across days?")
h21 = c_h[c_h['hour'] == 21]
by_date_h21 = h21.groupby('date')['went_up'].mean()
for date, up_pct in by_date_h21.items():
    dir_str = "UP bias" if up_pct > 0.55 else ("DOWN bias" if up_pct < 0.45 else "neutral")
    print(f"    {date}: {up_pct:.1%} ({dir_str})")

print(f"\n  Hourly bias consistency (std across days):")
hourly_consistency = c_h.groupby(['hour', 'date'])['went_up'].mean().groupby('hour').agg(['mean', 'std', 'count'])
hourly_consistency.columns = ['mean_up', 'std_up', 'n_days']
for hour, row in hourly_consistency.iterrows():
    if row['n_days'] >= 3:
        stable = "STABLE" if row['std_up'] < 0.08 else "variable"
        bias = "UP" if row['mean_up'] > 0.52 else ("DOWN" if row['mean_up'] < 0.48 else "neutral")
        marker = " ★" if row['std_up'] < 0.08 and abs(row['mean_up'] - 0.5) > 0.03 else ""
        print(f"    Hour {int(hour):>2}: mean={row['mean_up']:.1%} std={row['std_up']:.2f} ({stable}, {bias}){marker}")

# ═══════════════════════════════════════════════════════════════
# 6C: SEQUENCE PATTERNS — Markov chains
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("6C: Sequence Patterns — Transition Probabilities")
print(f"{'='*60}\n")

c_seq = trades.copy()
c_seq['d'] = c_seq['went_up'].astype(int)
c_seq['d1'] = c_seq['d'].shift(1)
c_seq['d2'] = c_seq['d'].shift(2)
c_seq['d3'] = c_seq['d'].shift(3)
c_seq = c_seq.dropna()

print("  2nd-order Markov (P(next | last 2)):")
for d2, d1 in [(0,0), (0,1), (1,0), (1,1)]:
    mask = (c_seq['d1'] == d1) & (c_seq['d2'] == d2)
    subset = c_seq[mask]
    if len(subset) > 50:
        p_up = subset['d'].mean()
        pattern = f"{'U' if d2 else 'D'}{'U' if d1 else 'D'}"
        best = max(p_up, 1-p_up)
        marker = " ★" if best > 0.52 else ""
        print(f"    {pattern} → P(UP)={p_up:.1%}, Best={best:.1%} ({len(subset)}){marker}")

print(f"\n  3rd-order Markov (P(next | last 3)):")
for d3 in [0, 1]:
    for d2 in [0, 1]:
        for d1 in [0, 1]:
            mask = (c_seq['d1'] == d1) & (c_seq['d2'] == d2) & (c_seq['d3'] == d3)
            subset = c_seq[mask]
            if len(subset) > 30:
                p_up = subset['d'].mean()
                pattern = f"{'U' if d3 else 'D'}{'U' if d2 else 'D'}{'U' if d1 else 'D'}"
                best = max(p_up, 1-p_up)
                marker = " ★" if best > 0.53 else ""
                print(f"    {pattern} → P(UP)={p_up:.1%}, Best={best:.1%} ({len(subset)}){marker}")

# ═══════════════════════════════════════════════════════════════
# 6D: MUTUAL INFORMATION
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("6D: Mutual Information — What features carry info?")
print(f"{'='*60}\n")

from sklearn.feature_selection import mutual_info_classif

c_mi = trades.copy()
c_mi['prev_dir'] = c_mi['went_up'].shift(1)
c_mi['prev2_dir'] = c_mi['went_up'].shift(2)
c_mi['prev3_dir'] = c_mi['went_up'].shift(3)
c_mi['prev_move'] = c_mi['move_abs'].shift(1)
c_mi['prev2_move'] = c_mi['move_abs'].shift(2)
c_mi['prev_move_signed'] = c_mi['move'].shift(1)

# Streak length — compute directly on c_mi
mi_streaks = []
mi_count = 1
mi_prev = None
for val in c_mi['went_up'].values:
    if val == mi_prev: mi_count += 1
    else: mi_count = 1
    mi_streaks.append(mi_count)
    mi_prev = val
c_mi['prev_streak'] = pd.Series(mi_streaks, index=c_mi.index).shift(1)

c_mi['price_vs_ma5'] = c_mi['close'] - c_mi['close'].rolling(5).mean()
c_mi['price_vs_ma20'] = c_mi['close'] - c_mi['close'].rolling(20).mean()
c_mi['minute_of_hour'] = c_mi.index.minute
c_mi = c_mi.dropna()

mi_features = ['prev_dir', 'prev2_dir', 'prev3_dir', 'prev_move', 'prev2_move',
                'prev_move_signed', 'hour', 'minute_of_hour', 'price_vs_ma5', 
                'price_vs_ma20', 'prev_streak']
mi_features = [f for f in mi_features if f in c_mi.columns]
X_mi = c_mi[mi_features].values.astype(float)
y_mi = c_mi['went_up'].values

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({'feature': mi_features, 'MI': mi_scores}).sort_values('MI', ascending=False)

for _, row in mi_df.iterrows():
    bar = '█' * int(row['MI'] * 5000)
    marker = " ★" if row['MI'] > 0.001 else ""
    print(f"  {row['feature']:<25} {row['MI']:>11.5f} {bar}{marker}")

print(f"\n  MI > 0.001 = meaningful info; MI ≈ 0 = nothing")

# ═══════════════════════════════════════════════════════════════
# 6E: BACKTEST — Streak-based strategy
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("6E: Backtest — Streak-Based Strategy")
print(f"{'='*60}\n")

for min_streak in [1, 2, 3, 4, 5]:
    mask = c_s['prev_streak'] >= min_streak
    subset = c_s[mask]
    preds = subset['prev_dir'].values
    actuals = subset['dir'].values
    r = backtest(preds, actuals)
    skip_pct = (1 - mask.mean()) * 100
    show_result(f"Follow when streak>={min_streak} (skip {skip_pct:.0f}%)", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7: HOURLY BIAS STRATEGY — The Most Consistent Signal
# ═══════════════════════════════════════════════════════════════

print("7: Hourly Bias Strategy — Exploit Fixed Directional Bias")
print("=" * 60)

c_hb = trades.copy()

# Learn hourly bias from TRAINING data (first 70%), test on remaining 30%
split = int(len(c_hb) * 0.7)
train = c_hb.iloc[:split]
test = c_hb.iloc[split:]

# Learn bias per hour from training data
hour_bias = train.groupby('hour')['went_up'].mean()
print(f"  Hourly UP bias (learned from training 70%):")
for hour, up_pct in hour_bias.items():
    bias = "UP" if up_pct > 0.52 else ("DOWN" if up_pct < 0.48 else "skip")
    marker = " ★" if abs(up_pct - 0.5) > 0.03 else ""
    print(f"    Hour {int(hour):>2}: {up_pct:.1%} → {bias}{marker}")

# Strategy A: Bet the bias direction for ALL hours
pred_all = test['hour'].map(lambda h: 1 if hour_bias.get(h, 0.5) > 0.5 else 0).values
actual_all = test['went_up'].values
acc_all = (pred_all == actual_all).mean()

# Strategy B: Only trade hours with strong bias (>53% or <47%)
strong_hours = [h for h, p in hour_bias.items() if abs(p - 0.5) > 0.03]
strong_mask = test['hour'].isin(strong_hours)
if strong_mask.sum() > 50:
    pred_strong = test.loc[strong_mask, 'hour'].map(
        lambda h: 1 if hour_bias.get(h, 0.5) > 0.5 else 0).values
    actual_strong = test.loc[strong_mask, 'went_up'].values
    acc_strong = (pred_strong == actual_strong).mean()
else:
    acc_strong = 0

# Strategy C: Only trade the MOST biased hours (>55% or <45%)
very_strong = [h for h, p in hour_bias.items() if abs(p - 0.5) > 0.05]
vs_mask = test['hour'].isin(very_strong)
if vs_mask.sum() > 50:
    pred_vs = test.loc[vs_mask, 'hour'].map(
        lambda h: 1 if hour_bias.get(h, 0.5) > 0.5 else 0).values
    actual_vs = test.loc[vs_mask, 'went_up'].values
    acc_vs = (pred_vs == actual_vs).mean()
else:
    acc_vs = 0

print(f"\n  OUT-OF-SAMPLE results (last 30% of data):")
print(f"    A) All hours, bet bias:          {acc_all:.1%} ({len(test)} trades)")
if strong_mask.sum() > 50:
    print(f"    B) Strong bias hours only:       {acc_strong:.1%} ({strong_mask.sum()} trades, {strong_mask.mean()*100:.0f}%)")
if vs_mask.sum() > 50:
    print(f"    C) Very strong bias only:        {acc_vs:.1%} ({vs_mask.sum()} trades, {vs_mask.mean()*100:.0f}%)")

# Backtest all strategies
print(f"\n  Martingale Backtests (out-of-sample):")
r = backtest(pred_all, actual_all)
show_result("A) All hours, bet bias", r)

if strong_mask.sum() > 100:
    r = backtest(pred_strong, actual_strong)
    show_result(f"B) Strong hours only ({len(strong_hours)}h)", r)

if vs_mask.sum() > 100:
    r = backtest(pred_vs, actual_vs)
    show_result(f"C) Very strong hours only", r)

# ═══════════════════════════════════════════════════════════════
# 8: COMBINED STRATEGY — Hourly Bias + Streak + Markov
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("8: Combined Strategy — Hourly Bias + Conditions")
print(f"{'='*60}\n")

# Use training data to learn everything
c_comb = trades.copy()

# Streak
comb_streaks = []
comb_count = 1
comb_prev = None
for val in c_comb['went_up'].values:
    if val == comb_prev: comb_count += 1
    else: comb_count = 1
    comb_streaks.append(comb_count)
    comb_prev = val
c_comb['streak'] = comb_streaks
c_comb['prev_streak'] = c_comb['streak'].shift(1)
c_comb['prev_dir'] = c_comb['went_up'].shift(1)
c_comb['prev2_dir'] = c_comb['went_up'].shift(2)
c_comb['prev3_dir'] = c_comb['went_up'].shift(3)
c_comb = c_comb.dropna()

split = int(len(c_comb) * 0.7)
train_c = c_comb.iloc[:split]
test_c = c_comb.iloc[split:]

# Learn hourly bias from training
hour_bias_train = train_c.groupby('hour')['went_up'].mean().to_dict()

# Combined strategies on TEST data
strategies_combined = []

# S1: Pure hourly bias
pred = test_c['hour'].map(lambda h: 1 if hour_bias_train.get(h, 0.5) > 0.5 else 0).values
r = backtest(pred, test_c['went_up'].values)
r['label'] = 'Hourly bias only'
strategies_combined.append(r)

# S2: Hourly bias, only trade biased hours
biased_hours = [h for h, p in hour_bias_train.items() if abs(p - 0.5) > 0.03]
mask = test_c['hour'].isin(biased_hours)
if mask.sum() > 100:
    pred = test_c.loc[mask, 'hour'].map(lambda h: 1 if hour_bias_train.get(h, 0.5) > 0.5 else 0).values
    r = backtest(pred, test_c.loc[mask, 'went_up'].values)
    r['label'] = f'Biased hours only (skip {(1-mask.mean())*100:.0f}%)'
    strategies_combined.append(r)

# S3: Hourly bias + follow when streak >= 3
pred_list = []
actual_list = []
for _, row in test_c.iterrows():
    h = int(row['hour'])
    streak = row['prev_streak']
    prev_d = row['prev_dir']
    
    # If streak >= 3, follow the streak direction
    if streak >= 3:
        pred_list.append(int(prev_d))
    else:
        # Otherwise use hourly bias
        pred_list.append(1 if hour_bias_train.get(h, 0.5) > 0.5 else 0)
    actual_list.append(int(row['went_up']))

r = backtest(pred_list, actual_list)
r['label'] = 'Hourly bias + streak>=3 follow'
strategies_combined.append(r)

# S4: Only trade when hourly bias AND streak agree
pred_list = []
actual_list = []
for _, row in test_c.iterrows():
    h = int(row['hour'])
    streak = row['prev_streak']
    prev_d = row['prev_dir']
    hour_pred = 1 if hour_bias_train.get(h, 0.5) > 0.5 else 0
    
    # Only trade when streak direction matches hourly bias
    if streak >= 2 and int(prev_d) == hour_pred:
        pred_list.append(hour_pred)
        actual_list.append(int(row['went_up']))

if len(pred_list) > 100:
    r = backtest(pred_list, actual_list)
    skip_pct = (1 - len(pred_list) / len(test_c)) * 100
    r['label'] = f'Hourly + streak agree (skip {skip_pct:.0f}%)'
    strategies_combined.append(r)

# S5: DUU → DOWN (the 53.1% Markov signal) + hourly bias
pred_list = []
actual_list = []
for _, row in test_c.iterrows():
    d1 = row['prev_dir']
    d2 = row['prev2_dir']
    d3 = row['prev3_dir']
    h = int(row['hour'])
    
    # DUU pattern → bet DOWN
    if d3 == 0 and d2 == 1 and d1 == 1:
        pred_list.append(0)  # DOWN
    else:
        # Hourly bias
        pred_list.append(1 if hour_bias_train.get(h, 0.5) > 0.5 else 0)
    actual_list.append(int(row['went_up']))

r = backtest(pred_list, actual_list)
r['label'] = 'DUU→DOWN + hourly bias'
strategies_combined.append(r)

# Results
strategies_combined.sort(key=lambda x: x['win_rate'], reverse=True)
print(f"  {'Strategy':<40} {'Win%':>6} {'Trades':>6} {'Busts':>6} {'Profit':>9}")
print(f"  {'-'*70}")
for r in strategies_combined:
    show_result(r['label'], r)

# Best strategy equity curve
fig, ax = plt.subplots(figsize=(16, 6))
for r in strategies_combined[:4]:
    ax.plot(r['history'], label=f"{r['label']} ({r['win_rate']:.1f}%)", linewidth=1.5)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Combined Strategies — Out of Sample')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9: MINUTE-OF-HOUR & FINER TIME PATTERNS
# ═══════════════════════════════════════════════════════════════

print("9: Minute-of-Hour Patterns (MI showed this carries info)")
print("=" * 60)

c_min = trades.copy()
c_min['minute'] = c_min.index.minute

# Is there a bias at specific minutes within the hour?
by_minute = c_min.groupby('minute').agg(
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(16, 5))
colors = ['lime' if abs(p - 0.5) > 0.03 else 'gray' for p in by_minute['up_pct']]
ax.bar(by_minute['minute'], by_minute['up_pct'], color=colors, alpha=0.7)
ax.axhline(y=0.5, color='white', linestyle='--', alpha=0.5)
ax.axhline(y=0.54, color='red', linestyle=':', alpha=0.5, label='54%')
ax.axhline(y=0.46, color='red', linestyle=':', alpha=0.5)
ax.set_title('UP Bias by Minute of Hour')
ax.set_xlabel('Minute')
ax.set_ylabel('P(UP)')
ax.legend()
plt.tight_layout()
plt.show()

# Which minutes are consistently biased?
print(f"\n  Minute-of-hour bias (all data):")
biased_minutes = []
for _, row in by_minute.iterrows():
    if abs(row['up_pct'] - 0.5) > 0.03:
        direction = "UP" if row['up_pct'] > 0.5 else "DOWN"
        biased_minutes.append((int(row['minute']), row['up_pct']))
        print(f"    Minute {int(row['minute']):>2}: {row['up_pct']:.1%} ({direction}) [{int(row['count'])} trades]")

# Check if minute bias is stable across different days
print(f"\n  Checking stability across days for biased minutes:")
c_min['date'] = c_min.index.date
for minute, overall_pct in biased_minutes:
    by_date = c_min[c_min['minute'] == minute].groupby('date')['went_up'].mean()
    std = by_date.std()
    direction = "UP" if overall_pct > 0.5 else "DOWN"
    consistent = sum(1 for p in by_date if (p > 0.5) == (overall_pct > 0.5))
    print(f"    Min {minute:>2}: {overall_pct:.1%} {direction}, std={std:.2f}, "
          f"consistent {consistent}/{len(by_date)} days")

# ═══════════════════════════════════════════════════════════════
# 10: 5-MINUTE BLOCKS — Coarser Time Patterns
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("10: 5-Minute Block Patterns")
print(f"{'='*60}\n")

c_min['block_5'] = c_min['minute'] // 5  # 0-11 (twelve 5-minute blocks per hour)
c_min['time_slot'] = c_min['hour'] * 12 + c_min['block_5']  # 0-287

# Bias by 5-minute block (across all hours)
by_block = c_min.groupby('block_5').agg(
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()

print("  UP bias by 5-minute block within hour:")
for _, row in by_block.iterrows():
    mins = f":{int(row['block_5'])*5:02d}-:{int(row['block_5'])*5+4:02d}"
    marker = " ★" if abs(row['up_pct'] - 0.5) > 0.02 else ""
    print(f"    Block {int(row['block_5']):>2} ({mins}): {row['up_pct']:.1%} ({int(row['count'])} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# 11: ABSOLUTE PRICE LEVEL — Round numbers, specific zones
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("11: Price Level Analysis")
print(f"{'='*60}\n")

c_price = trades.copy()

# Distance from round number (e.g., 1.14000, 1.14100, etc.)
c_price['price_mod_100'] = (c_price['close'] * 100000) % 100  # Pips within 100-pip range
c_price['near_round'] = ((c_price['price_mod_100'] < 10) | (c_price['price_mod_100'] > 90)).astype(int)

# Near round numbers
for label, mask in [('Near round number (±10 pips)', c_price['near_round'] == 1),
                     ('Away from round number', c_price['near_round'] == 0)]:
    subset = c_price[mask]
    if len(subset) > 100:
        up_pct = subset['went_up'].mean()
        follow_acc = (subset['went_up'] == subset['prev_went_up']).mean()
        print(f"  {label:<35}: UP={up_pct:.1%}, Follow={follow_acc:.1%} ({len(subset)} trades)")

# Price position within the day's range
c_price['day_high'] = c_price.groupby(c_price.index.date)['close'].transform('max')
c_price['day_low'] = c_price.groupby(c_price.index.date)['close'].transform('min')
c_price['day_range'] = c_price['day_high'] - c_price['day_low']
c_price['day_position'] = np.where(c_price['day_range'] > 0,
    (c_price['close'] - c_price['day_low']) / c_price['day_range'], 0.5)

print(f"\n  Price position within day's range:")
for label, lo, hi in [('Bottom 20%', 0, 0.2), ('20-40%', 0.2, 0.4), 
                        ('40-60% (middle)', 0.4, 0.6), ('60-80%', 0.6, 0.8), 
                        ('Top 20%', 0.8, 1.01)]:
    mask = (c_price['day_position'] >= lo) & (c_price['day_position'] < hi)
    subset = c_price[mask]
    if len(subset) > 100:
        up_pct = subset['went_up'].mean()
        marker = " ★" if abs(up_pct - 0.5) > 0.03 else ""
        print(f"    {label:<20}: P(UP)={up_pct:.1%} ({len(subset)} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# 12: COMPREHENSIVE ML — All signals that showed any life
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("12: ML with ALL Signals That Showed Any Life")
print(f"{'='*60}\n")

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

c_all = trades.copy()

# Compute all features
# Streaks
all_streaks = []
all_count = 1
all_prev = None
for val in c_all['went_up'].values:
    if val == all_prev: all_count += 1
    else: all_count = 1
    all_streaks.append(all_count)
    all_prev = val
c_all['streak'] = all_streaks
c_all['prev_streak'] = c_all['streak'].shift(1)
c_all['prev_dir'] = c_all['went_up'].shift(1)
c_all['prev2_dir'] = c_all['went_up'].shift(2)
c_all['prev3_dir'] = c_all['went_up'].shift(3)
c_all['prev_move'] = c_all['move_abs'].shift(1)
c_all['prev_move_signed'] = c_all['move'].shift(1)
c_all['minute'] = c_all.index.minute
c_all['block_5'] = c_all['minute'] // 5

# Price position
c_all['ma5'] = c_all['close'].rolling(5).mean()
c_all['ma20'] = c_all['close'].rolling(20).mean()
c_all['price_vs_ma5'] = c_all['close'] - c_all['ma5']
c_all['price_vs_ma20'] = c_all['close'] - c_all['ma20']

c_all = c_all.dropna()

# ALL causal features
all_features = ['prev_dir', 'prev2_dir', 'prev3_dir', 'prev_streak', 
                'prev_move', 'prev_move_signed',
                'hour', 'minute', 'block_5',
                'price_vs_ma5', 'price_vs_ma20']

X = c_all[all_features].values.astype(float)
y = c_all['went_up'].values

# Cross-validation
gb = GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, 
                                 subsample=0.8, random_state=42)
scores = cross_val_score(gb, X, y, cv=5, scoring='accuracy')
print(f"  Gradient Boosting (all signals): {scores.mean():.1%} (±{scores.std():.1%})")

# Out-of-sample
split = int(len(X) * 0.7)
gb.fit(X[:split], y[:split])
pred = gb.predict(X[split:])
oos = (pred == y[split:]).mean()
print(f"  Out-of-sample:                   {oos:.1%}")

# Backtest
r = backtest(pred, y[split:])
show_result("GB all signals (out-of-sample)", r)

# Feature importance
importances = pd.DataFrame({
    'feature': all_features,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n  Feature importances:")
for _, row in importances.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"    {row['feature']:<20} {row['importance']:.3f} {bar}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 13: FUNDAMENTALLY DIFFERENT APPROACHES
# ═══════════════════════════════════════════════════════════════
# Stop asking "does direction predict direction?"
# Instead ask: what IS the generative process?

print("13: Reverse-Engineering the OTC Price Generator")
print("=" * 60)

# ── 13A: Is the sequence TRULY random? Run-length test ──
print("\n13A: Run-Length Distribution — Random Walk vs OTC")
print("-" * 60)

# In a fair coin flip, run lengths follow geometric distribution
# P(run of length k) = 0.5^k
# Expected mean run length = 2

directions = trades['went_up'].values
runs = []
current_len = 1
for i in range(1, len(directions)):
    if directions[i] == directions[i-1]:
        current_len += 1
    else:
        runs.append(current_len)
        current_len = 1
runs.append(current_len)

# Compare to random
random_dirs = np.random.randint(0, 2, size=len(directions))
random_runs = []
current_len = 1
for i in range(1, len(random_dirs)):
    if random_dirs[i] == random_dirs[i-1]:
        current_len += 1
    else:
        random_runs.append(current_len)
        current_len = 1
random_runs.append(current_len)

print(f"  OTC data:     mean run={np.mean(runs):.2f}, median={np.median(runs):.0f}, max={max(runs)}")
print(f"  Random walk:  mean run={np.mean(random_runs):.2f}, median={np.median(random_runs):.0f}, max={max(random_runs)}")
print(f"  Expected (geometric): mean=2.0, median=1.0")

# Chi-squared test: does run-length distribution match geometric?
from scipy.stats import chisquare
max_run = 10
otc_hist = np.array([sum(1 for r in runs if r == k) for k in range(1, max_run + 1)])
expected_hist = np.array([len(runs) * 0.5**k for k in range(1, max_run + 1)])
# Normalize expected to match total
expected_hist = expected_hist * otc_hist.sum() / expected_hist.sum()

chi2, p_chi = chisquare(otc_hist, expected_hist)
print(f"\n  Chi-squared vs geometric: χ²={chi2:.2f}, p={p_chi:.4f}")
print(f"  → {'Run lengths DIFFER from random!' if p_chi < 0.05 else 'Run lengths match random'}")

fig, ax = plt.subplots(figsize=(12, 5))
x = range(1, max_run + 1)
ax.bar([xi - 0.15 for xi in x], otc_hist / otc_hist.sum(), width=0.3, label='OTC', color='cyan', alpha=0.7)
ax.bar([xi + 0.15 for xi in x], expected_hist / expected_hist.sum(), width=0.3, label='Random (geometric)', color='red', alpha=0.5)
ax.set_title('Run-Length Distribution: OTC vs Random')
ax.set_xlabel('Run length (consecutive same direction)')
ax.set_ylabel('Proportion')
ax.legend()
plt.tight_layout()
plt.show()

# ── 13B: Are UP and DOWN moves symmetric? ──
print("\n13B: UP vs DOWN Move Symmetry")
print("-" * 60)

up_moves = trades[trades['went_up'] == 1]['move_abs']
down_moves = trades[trades['went_up'] == 0]['move_abs']

print(f"  UP moves:   mean={up_moves.mean():.6f}, median={up_moves.median():.6f}, n={len(up_moves)}")
print(f"  DOWN moves: mean={down_moves.mean():.6f}, median={down_moves.median():.6f}, n={len(down_moves)}")

from scipy.stats import mannwhitneyu
stat, p_mw = mannwhitneyu(up_moves, down_moves)
print(f"  Mann-Whitney U test: p={p_mw:.4f}")
print(f"  → {'UP and DOWN move sizes DIFFER!' if p_mw < 0.05 else 'Symmetric — no size difference'}")

# ── 13C: Autocorrelation of SIGNED returns ──
print("\n13C: Autocorrelation of Signed Returns (not just direction)")
print("-" * 60)

returns = trades['move'].values
for lag in range(1, 11):
    corr, pval = pearsonr(returns[lag:], returns[:-lag])
    sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
    print(f"  Lag {lag:>2}: r={corr:+.4f} (p={pval:.4f}) {sig}")

# ── 13D: Does the MAGNITUDE of previous move predict next direction? ──
print("\n13D: Previous Move Magnitude → Next Direction")
print("-" * 60)

c_mag = trades.copy()
c_mag['prev_move_signed'] = c_mag['move'].shift(1)
c_mag['prev_move_abs'] = c_mag['move_abs'].shift(1)
c_mag = c_mag.dropna()

# Decile analysis
c_mag['move_decile'] = pd.qcut(c_mag['prev_move_signed'], q=10, duplicates='drop')
by_decile = c_mag.groupby('move_decile').agg(
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()

print(f"  {'Previous Move Range':<30} {'P(Next UP)':>10} {'N':>6}")
print(f"  {'-'*50}")
for _, row in by_decile.iterrows():
    marker = " ★" if abs(row['up_pct'] - 0.5) > 0.03 else ""
    print(f"  {str(row['move_decile']):<30} {row['up_pct']:>9.1%} {int(row['count']):>6}{marker}")

# ── 13E: Fourier / Spectral Analysis — periodic patterns ──
print("\n13E: Spectral Analysis — Are there periodic patterns?")
print("-" * 60)

from scipy.fft import fft

# FFT of direction sequence
signal = directions.astype(float) - 0.5  # Center around 0
n = len(signal)
freqs = np.fft.fftfreq(n)
spectrum = np.abs(fft(signal))[:n//2]
freqs_pos = freqs[:n//2]

# Convert frequency to period in minutes
periods = np.where(freqs_pos > 0, 1.0 / freqs_pos, np.inf)

# Find dominant periods (excluding DC)
top_indices = np.argsort(spectrum[1:])[-10:] + 1
print(f"  Top 10 dominant periods in direction sequence:")
for idx in reversed(top_indices):
    period = periods[idx]
    power = spectrum[idx]
    if 2 <= period <= 1000:
        print(f"    Period: {period:.1f} minutes, Power: {power:.1f}")

# Plot spectrum for periods 2-100 minutes
mask = (periods >= 2) & (periods <= 200)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(periods[mask], spectrum[mask], color='cyan', linewidth=0.5)
ax.set_title('Spectral Analysis of Direction Sequence')
ax.set_xlabel('Period (minutes)')
ax.set_ylabel('Spectral Power')
ax.set_xscale('log')
plt.tight_layout()
plt.show()

# ── 13F: Hidden Markov Model — regime detection ──
print("\n13F: Hidden State Detection — Do Regimes Exist?")
print("-" * 60)

# Simple HMM: are there "UP regime" and "DOWN regime" states?
# Test: split data into blocks and check if UP% varies significantly
block_size = 30  # 30-minute blocks
n_blocks = len(directions) // block_size
block_up_pcts = []
for i in range(n_blocks):
    block = directions[i * block_size:(i + 1) * block_size]
    block_up_pcts.append(block.mean())

block_up_pcts = np.array(block_up_pcts)

# If truly random: each block should be ~50% with std = sqrt(0.25/30) = 0.091
expected_std = np.sqrt(0.25 / block_size)
actual_std = block_up_pcts.std()

print(f"  30-minute block analysis ({n_blocks} blocks):")
print(f"    Mean UP%:     {block_up_pcts.mean():.1%}")
print(f"    Std of UP%:   {actual_std:.3f}")
print(f"    Expected std: {expected_std:.3f} (if random)")
print(f"    Ratio:        {actual_std / expected_std:.2f}x")

if actual_std > expected_std * 1.2:
    print(f"    → EXCESS VARIANCE detected! Blocks vary more than random.")
    print(f"      This suggests regime-like behavior (UP periods, DOWN periods)")
    
    # How many blocks are significantly biased?
    sig_up = sum(1 for p in block_up_pcts if p > 0.5 + 2 * expected_std)
    sig_down = sum(1 for p in block_up_pcts if p < 0.5 - 2 * expected_std)
    print(f"      Significantly UP blocks: {sig_up}/{n_blocks} ({sig_up/n_blocks*100:.0f}%)")
    print(f"      Significantly DOWN blocks: {sig_down}/{n_blocks} ({sig_down/n_blocks*100:.0f}%)")
else:
    print(f"    → Variance matches random. No regime-like behavior.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 14: UNEXPLORED DATA — Volume, Price Digits, Sub-Second
# ═══════════════════════════════════════════════════════════════

print("14: Data We Haven't Used Yet")
print("=" * 60)

# ── 14A: Volume Analysis ──
print("\n14A: Volume — Does trading activity predict direction?")
print("-" * 60)

if 'volume' in raw.columns:
    # Get volume at :00 candle and sum of volume within each minute
    c_vol = trades.copy()
    
    # Sum volume per minute
    raw_with_min = raw.copy()
    raw_with_min['minute_key'] = raw_with_min.index.floor('min')
    vol_per_minute = raw_with_min.groupby('minute_key')['volume'].sum()
    c_vol['minute_vol'] = vol_per_minute.reindex(c_vol.index)
    c_vol['prev_vol'] = c_vol['minute_vol'].shift(1)
    c_vol = c_vol.dropna(subset=['prev_vol', 'minute_vol'])
    
    print(f"  Volume stats: mean={c_vol['minute_vol'].mean():.0f}, "
          f"median={c_vol['minute_vol'].median():.0f}, "
          f"std={c_vol['minute_vol'].std():.0f}")
    
    # Does volume predict direction?
    if c_vol['prev_vol'].std() > 0:
        c_vol['prev_vol_q'] = pd.qcut(c_vol['prev_vol'], q=5, duplicates='drop',
                                        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        
        print(f"\n  Direction by previous minute's volume:")
        by_vol = c_vol.groupby('prev_vol_q').agg(
            up_pct=('went_up', 'mean'),
            follow_acc=('went_up', lambda x: (x == c_vol.loc[x.index, 'prev_went_up']).mean()),
            count=('went_up', 'count'),
        ).reset_index()
        
        for _, row in by_vol.iterrows():
            marker = " ★" if abs(row['up_pct'] - 0.5) > 0.03 or row['follow_acc'] > 0.52 else ""
            print(f"    {row['prev_vol_q']:<12}: UP={row['up_pct']:.1%}, Follow={row['follow_acc']:.1%} ({int(row['count'])}){marker}")
        
        # Volume autocorrelation (does high volume predict high volume?)
        v_corr = c_vol['minute_vol'].corr(c_vol['prev_vol'])
        print(f"\n  Volume autocorrelation: r={v_corr:.3f}")
        
        # High volume + direction
        c_vol['vol_spike'] = (c_vol['prev_vol'] > c_vol['prev_vol'].quantile(0.9)).astype(int)
        if c_vol['vol_spike'].sum() > 50:
            spike_acc = (c_vol.loc[c_vol['vol_spike'] == 1, 'went_up'] == 
                        c_vol.loc[c_vol['vol_spike'] == 1, 'prev_went_up']).mean()
            print(f"  After volume spike (top 10%): follow={spike_acc:.1%} ({c_vol['vol_spike'].sum()} trades)")
    else:
        print("  Volume is constant — no variation to analyze")
else:
    print("  No volume column in data")

# ── 14B: Price Precision & Last Digit Patterns ──
print(f"\n{'='*60}")
print("14B: Price Precision — Last Digit Patterns")
print("-" * 60)

# What precision does the OTC price use?
prices = trades['close'].values
# Count decimal places
sample_price = prices[0]
print(f"  Sample price: {sample_price}")
print(f"  As string: {str(sample_price)}")

# Last digit distribution (in 5th decimal place / half-pip)
last_digit = np.round(prices * 100000) % 10
last_digit = last_digit.astype(int)

digit_counts = pd.Series(last_digit).value_counts().sort_index()
print(f"\n  Last digit (5th decimal) distribution:")
for digit, count in digit_counts.items():
    pct = count / len(last_digit) * 100
    bar = '█' * int(pct * 2)
    expected = 10.0  # 10% each if uniform
    marker = " ★" if abs(pct - expected) > 2 else ""
    print(f"    Digit {digit}: {pct:.1f}% ({count}){marker} {bar}")

# Does the last digit predict direction?
print(f"\n  Direction by last digit:")
c_digit = trades.copy()
c_digit['last_digit'] = (np.round(c_digit['close'] * 100000) % 10).astype(int)
by_digit = c_digit.groupby('last_digit')['went_up'].mean()
for digit, up_pct in by_digit.items():
    marker = " ★" if abs(up_pct - 0.5) > 0.03 else ""
    print(f"    Digit {digit}: P(UP)={up_pct:.1%}{marker}")

# Last 2 digits
c_digit['last_2'] = (np.round(c_digit['close'] * 100000) % 100).astype(int)
# Even vs odd
c_digit['is_even'] = (c_digit['last_digit'] % 2 == 0).astype(int)
even_up = c_digit[c_digit['is_even'] == 1]['went_up'].mean()
odd_up = c_digit[c_digit['is_even'] == 0]['went_up'].mean()
print(f"\n  Even last digit: P(UP)={even_up:.1%}")
print(f"  Odd last digit:  P(UP)={odd_up:.1%}")

# ── 14C: Price Changes at Exact Boundaries ──
print(f"\n{'='*60}")
print("14C: What Happens at Exact :00 Boundary?")
print("-" * 60)

# The :59 close vs :00 open — is there a jump?
c00_raw = raw[raw['second'] == 0][['open', 'close']].copy()
c59_raw = raw[raw['second'] == 59][['close']].copy()
c59_raw.index = c59_raw.index + pd.Timedelta(seconds=1)
c00_raw['prev_59_close'] = c59_raw['close']
c00_raw = c00_raw.dropna()

gap = c00_raw['open'] - c00_raw['prev_59_close']
print(f"  Gap (:59 close → :00 open):")
print(f"    Mean:   {gap.mean():.7f}")
print(f"    Median: {gap.median():.7f}")
print(f"    Std:    {gap.std():.7f}")
print(f"    % positive gaps: {(gap > 0).mean():.1%}")
print(f"    % zero gaps:     {(gap == 0).mean():.1%}")
print(f"    % negative gaps: {(gap < 0).mean():.1%}")

# Does gap direction predict minute direction?
gap_up = (gap > 0).astype(int)
minute_up = (c00_raw['close'].shift(-1) > c00_raw['close']).astype(int)  # Use shift(-1) = next :00 close
valid_gap = pd.DataFrame({'gap_up': gap_up, 'minute_up': minute_up}).dropna()
gap_follow = (valid_gap['minute_up'] == valid_gap['gap_up']).mean()
gap_fade = (valid_gap['minute_up'] != valid_gap['gap_up']).mean()
print(f"\n  Gap direction predicts minute:")
print(f"    Follow gap: {gap_follow:.1%}")
print(f"    Fade gap:   {gap_fade:.1%}")

# ── 14D: Intra-Second Patterns (OHLC within 1s candles) ──
print(f"\n{'='*60}")
print("14D: Intra-Second OHLC — Wick Analysis")
print("-" * 60)

c00_ohlc = raw[raw['second'] == 0][['open', 'high', 'low', 'close']].copy()
c00_ohlc['body'] = c00_ohlc['close'] - c00_ohlc['open']
c00_ohlc['upper_wick'] = c00_ohlc['high'] - c00_ohlc[['open', 'close']].max(axis=1)
c00_ohlc['lower_wick'] = c00_ohlc[['open', 'close']].min(axis=1) - c00_ohlc['low']
c00_ohlc['range'] = c00_ohlc['high'] - c00_ohlc['low']

# What's the :00 candle typically like?
print(f"  :00 candle (1-second) characteristics:")
print(f"    Avg body:       {c00_ohlc['body'].abs().mean():.7f}")
print(f"    Avg range:      {c00_ohlc['range'].mean():.7f}")
print(f"    Avg upper wick: {c00_ohlc['upper_wick'].mean():.7f}")
print(f"    Avg lower wick: {c00_ohlc['lower_wick'].mean():.7f}")
print(f"    % bullish:      {(c00_ohlc['body'] > 0).mean():.1%}")
print(f"    % doji (O==C):  {(c00_ohlc['body'] == 0).mean():.1%}")

# Does the :00 candle's wick pattern predict the minute?
c00_ohlc['minute_up'] = (c00_ohlc['close'].shift(-1) > c00_ohlc['close']).astype(int)
c00_ohlc['big_upper_wick'] = (c00_ohlc['upper_wick'] > c00_ohlc['range'] * 0.5).astype(int)
c00_ohlc['big_lower_wick'] = (c00_ohlc['lower_wick'] > c00_ohlc['range'] * 0.5).astype(int)

valid = c00_ohlc.dropna(subset=['minute_up'])
for label, mask in [('Big upper wick (sellers)', valid['big_upper_wick'] == 1),
                     ('Big lower wick (buyers)', valid['big_lower_wick'] == 1),
                     ('Bullish :00 candle', valid['body'] > 0),
                     ('Bearish :00 candle', valid['body'] < 0)]:
    if mask.sum() > 100:
        up_pct = valid.loc[mask, 'minute_up'].mean()
        marker = " ★" if abs(up_pct - 0.5) > 0.03 else ""
        print(f"    {label:<30}: P(UP)={up_pct:.1%} ({mask.sum()} trades){marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 15: EXPLOIT THE PRICE GENERATOR'S STRUCTURE
# ═══════════════════════════════════════════════════════════════

print("15: Exploiting the OTC Price Generator")
print("=" * 60)

# The generator uses 0.00002 steps (even digits = 82% of prices)
# Odd digits are "transitional" — what happens after them?

print("\n15A: Odd vs Even Digit — Transitional States")
print("-" * 60)

c_gen = trades.copy()
c_gen['last_digit'] = (np.round(c_gen['close'] * 100000) % 10).astype(int)
c_gen['is_odd'] = (c_gen['last_digit'] % 2 == 1).astype(int)
c_gen['prev_is_odd'] = c_gen['is_odd'].shift(1)

# When previous close is on odd digit, is direction more predictable?
c_gen = c_gen.dropna()
for label, mask in [('After EVEN digit (normal)', c_gen['prev_is_odd'] == 0),
                     ('After ODD digit (transitional)', c_gen['prev_is_odd'] == 1)]:
    sub = c_gen[mask]
    follow = (sub['went_up'] == sub['prev_went_up']).mean()
    up = sub['went_up'].mean()
    print(f"  {label}: P(UP)={up:.1%}, Follow={follow:.1%} ({len(sub)} trades)")

# ── 15B: Step size analysis ──
print(f"\n15B: Price Step Sizes")
print("-" * 60)

# What are the actual step sizes between consecutive :00 closes?
steps = np.diff(trades['close'].values)
step_pips = np.round(steps * 100000).astype(int)  # In 0.00001 units

print(f"  Step size distribution (in 0.00001 units):")
step_counts = pd.Series(step_pips).value_counts().sort_index()
for step, count in step_counts.head(20).items():
    if count > 50:
        pct = count / len(step_pips) * 100
        print(f"    Step {step:>4}: {pct:.1f}% ({count})")

# Is the step ALWAYS even?
even_steps = (np.abs(step_pips) % 2 == 0).mean()
print(f"\n  % even step sizes: {even_steps:.1%}")
print(f"  % odd step sizes:  {1-even_steps:.1%}")

# ── 15C: Does step SIZE predict next direction? ──
print(f"\n15C: Step Size → Next Direction")
print("-" * 60)

c_step = trades.copy()
c_step['step'] = c_step['close'].diff() * 100000  # In pip units
c_step['prev_step'] = c_step['step'].shift(1)
c_step['prev_step_abs'] = c_step['prev_step'].abs()
c_step = c_step.dropna()

# Group by step size
c_step['step_size_cat'] = pd.cut(c_step['prev_step_abs'], 
                                   bins=[0, 1, 2, 5, 10, 20, 50, 1000],
                                   labels=['<1pip', '1-2', '2-5', '5-10', '10-20', '20-50', '50+'])

by_step = c_step.groupby('step_size_cat').agg(
    up_pct=('went_up', 'mean'),
    follow=('went_up', lambda x: (x == c_step.loc[x.index, 'prev_went_up']).mean()),
    count=('went_up', 'count'),
).reset_index()

for _, row in by_step.iterrows():
    marker = " ★" if abs(row['up_pct'] - 0.5) > 0.03 or row['follow'] > 0.52 else ""
    print(f"  Prev step {row['step_size_cat']:<8}: UP={row['up_pct']:.1%}, Follow={row['follow']:.1%} ({int(row['count'])}){marker}")

# ── 15D: The :00 Mean Reversion — Deeper Dive ──
print(f"\n{'='*60}")
print("15D: Mean Reversion at :00 — Can We Trade This?")
print("-" * 60)

# Bullish :00 candle → 48% UP = 52% DOWN (fade)
# If we FADE the first second, enter at :01, what's the accuracy?

c_rev = trades.dropna(subset=['c_01', 'c_02', 'c_03']).copy()

# The :00 candle: open to close in first second
c_rev['sec0_up'] = (c_rev['close'] > c_rev['open']).astype(int)  # :00 candle direction
c_rev['sec0_move'] = (c_rev['close'] - c_rev['open']).abs()

# Trade: enter at :01, result at next :00
# Fade the :00 candle direction
c_rev['trade_result'] = (c_rev['result_close'] > c_rev['c_01']).astype(int)
c_rev['fade_signal'] = 1 - c_rev['sec0_up']  # Opposite of first second

fade_acc = (c_rev['trade_result'] == c_rev['fade_signal']).mean()
follow_acc = (c_rev['trade_result'] == c_rev['sec0_up']).mean()

print(f"  Enter at :01, use :00 candle direction:")
print(f"    Follow :00 candle: {follow_acc:.1%}")
print(f"    Fade :00 candle:   {fade_acc:.1%}")

# Does the size of the :00 candle matter?
print(f"\n  Conditional on :00 candle size:")
c_rev['sec0_size_q'] = pd.qcut(c_rev['sec0_move'], q=4, labels=['Tiny', 'Small', 'Medium', 'Large'], duplicates='drop')
for size in ['Tiny', 'Small', 'Medium', 'Large']:
    mask = c_rev['sec0_size_q'] == size
    if mask.sum() > 100:
        fade = (c_rev.loc[mask, 'trade_result'] != c_rev.loc[mask, 'sec0_up']).mean()
        print(f"    {size:<8} :00 candle: Fade={fade:.1%} ({mask.sum()} trades)")

# Extend: what about fading the first 2-3 seconds?
for n_sec in [1, 2, 3, 5]:
    col = f'c_{n_sec:02d}'
    if col in c_rev.columns:
        initial_up = (c_rev[col] > c_rev['close']).astype(int)  # :00 to :N direction
        trade_up = (c_rev['result_close'] > c_rev[col]).astype(int)  # :N to next :00
        fade = (trade_up != initial_up).mean()
        follow = (trade_up == initial_up).mean()
        print(f"    Fade first {n_sec}s (:0{n_sec}→:00): {fade:.1%} | Follow: {follow:.1%}")

# ── 15E: Combined — Fade :00 + Hourly Bias ──
print(f"\n{'='*60}")
print("15E: Combine Fade-:00 with Hourly Bias")
print("-" * 60)

# Maybe fading works better at biased hours
c_combo = c_rev.copy()
c_combo['hour'] = c_combo.index.hour

# Split train/test
split = int(len(c_combo) * 0.7)
train = c_combo.iloc[:split]
test = c_combo.iloc[split:]

# Learn which hours have strong fade signal
hour_fade = train.groupby('hour').apply(
    lambda g: (g['trade_result'] != g['sec0_up']).mean()
).to_dict()

print(f"  Fade :00 accuracy by hour (training data):")
strong_fade_hours = []
for h in sorted(hour_fade.keys()):
    acc = hour_fade[h]
    marker = " ★" if acc > 0.53 else ""
    if acc > 0.53:
        strong_fade_hours.append(int(h))
    print(f"    Hour {int(h):>2}: {acc:.1%}{marker}")

# Test: fade only during strong hours
if strong_fade_hours:
    mask = test['hour'].isin(strong_fade_hours)
    if mask.sum() > 50:
        fade_pred = 1 - test.loc[mask, 'sec0_up'].values
        actual = test.loc[mask, 'trade_result'].values
        acc = (fade_pred == actual).mean()
        r = backtest(fade_pred, actual)
        print(f"\n  OOS: Fade at strong hours ({strong_fade_hours}):")
        print(f"    Accuracy: {acc:.1%} ({mask.sum()} trades)")
        show_result(f"Fade :00 at strong hours", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 16: NEURAL NETWORK ON RAW PRICE SEQUENCES
# ═══════════════════════════════════════════════════════════════
# Let the network find patterns in the raw 60-second trajectory
# that our hand-crafted features miss.

print("16: Neural Network on Raw 1-Second Price Sequences")
print("=" * 60)

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Build sequences: for each minute, the PREVIOUS minute's 60-second price path
# Normalized as returns relative to :00 price

c_nn = trades.copy()
sec_cols = [f'c_{s:02d}' for s in range(1, 60)]
available_secs = [c for c in sec_cols if c in c_nn.columns and c_nn[c].notna().mean() > 0.9]
c_nn = c_nn.dropna(subset=available_secs[:30])  # Need at least 30 seconds

print(f"  Available seconds per minute: {len(available_secs)}")

# Build feature matrix: previous minute's price path (normalized)
X_rows = []
y_list = []

price_cols_nn = ['close'] + available_secs
for i in range(1, len(c_nn)):
    prev_row = c_nn.iloc[i - 1]
    curr_row = c_nn.iloc[i]
    
    # Previous minute's price path, normalized to start at 0
    prev_prices = [prev_row[c] for c in price_cols_nn if not pd.isna(prev_row[c])]
    if len(prev_prices) < 30:
        continue
    
    base = prev_prices[0]
    normalized = [(p - base) * 100000 for p in prev_prices]  # In 0.00001 units
    
    # Pad/truncate to fixed length
    target_len = 60
    if len(normalized) < target_len:
        normalized = normalized + [normalized[-1]] * (target_len - len(normalized))
    else:
        normalized = normalized[:target_len]
    
    X_rows.append(normalized)
    y_list.append(int(curr_row['went_up']))

X_nn = np.array(X_rows)
y_nn = np.array(y_list)

print(f"  Sequences: {X_nn.shape[0]}, Features per sequence: {X_nn.shape[1]}")

# Scale
scaler_nn = StandardScaler()
X_nn_scaled = scaler_nn.fit_transform(X_nn)

# ── MLP on raw sequences ──
print(f"\n  MLP Neural Network (2 hidden layers):")
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, 
                     early_stopping=True, validation_fraction=0.1)
mlp_scores = cross_val_score(mlp, X_nn_scaled, y_nn, cv=5, scoring='accuracy')
print(f"    5-fold CV: {mlp_scores.mean():.1%} (±{mlp_scores.std():.1%})")

# ── MLP with additional hand-crafted features ──
# Add summary stats to the raw sequence
X_extra = np.column_stack([
    X_nn_scaled,
    X_nn[:, -1],                          # Final price
    X_nn[:, -1] - X_nn[:, 0],            # Net change
    np.max(X_nn, axis=1),                 # Max price
    np.min(X_nn, axis=1),                 # Min price
    np.std(X_nn, axis=1),                 # Volatility
    (np.diff(X_nn, axis=1) > 0).sum(axis=1),  # Up-step count
])

mlp2 = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42,
                      early_stopping=True, validation_fraction=0.1)
mlp2_scores = cross_val_score(mlp2, X_extra, y_nn, cv=5, scoring='accuracy')
print(f"    MLP + summary features: {mlp2_scores.mean():.1%} (±{mlp2_scores.std():.1%})")

# ── Out of sample ──
split = int(len(X_nn) * 0.7)
mlp.fit(X_nn_scaled[:split], y_nn[:split])
pred_nn = mlp.predict(X_nn_scaled[split:])
oos_nn = (pred_nn == y_nn[split:]).mean()

print(f"\n  Out-of-sample: {oos_nn:.1%}")

r = backtest(pred_nn, y_nn[split:])
show_result("MLP on raw sequences (OOS)", r)

# ═══════════════════════════════════════════════════════════════
# 17: ORDINAL PATTERNS — Permutation Structure
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("17: Ordinal Patterns — Does Price Sequence Shape Predict?")
print(f"{'='*60}\n")

# Convert each 5-second window into an ordinal pattern
# (which of the 5 seconds had the highest, 2nd highest, etc. price)
# Then check if specific ordinal patterns predict next direction

c_ord = trades.dropna(subset=[f'c_{s:02d}' for s in [10, 20, 30, 40, 50]]).copy()

# Build 6-point ordinal pattern from :00, :10, :20, :30, :40, :50
pattern_cols = ['close', 'c_10', 'c_20', 'c_30', 'c_40', 'c_50']
patterns = c_ord[pattern_cols].values

# Compute ordinal pattern: rank of each point (0=lowest, 5=highest)
from scipy.stats import rankdata
ordinal = np.array([rankdata(row) for row in patterns])

# Convert to string pattern for grouping
c_ord['ordinal'] = [''.join(str(int(x)) for x in row) for row in ordinal]
c_ord['prev_ordinal'] = c_ord['ordinal'].shift(1)
c_ord = c_ord.dropna()

# How many unique ordinal patterns?
unique_patterns = c_ord['prev_ordinal'].nunique()
print(f"  Unique ordinal patterns: {unique_patterns}")

# For each pattern, what's P(UP)?
by_ord = c_ord.groupby('prev_ordinal').agg(
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()
by_ord = by_ord[by_ord['count'] >= 20].sort_values('up_pct', ascending=False)

print(f"  Patterns with strongest bias (n≥20):")
for _, row in by_ord.head(5).iterrows():
    print(f"    {row['prev_ordinal']}: P(UP)={row['up_pct']:.1%} ({int(row['count'])} trades) ★")
for _, row in by_ord.tail(5).iterrows():
    print(f"    {row['prev_ordinal']}: P(UP)={row['up_pct']:.1%} ({int(row['count'])} trades) ★")

# Overall: can ordinal patterns predict?
best_per_pattern = by_ord.apply(lambda r: max(r['up_pct'], 1 - r['up_pct']), axis=1)
weighted_best = (best_per_pattern * by_ord['count']).sum() / by_ord['count'].sum()
print(f"\n  Weighted best accuracy using ordinal patterns: {weighted_best:.1%}")
print(f"  (If >54%, ordinal patterns carry tradeable signal)")

# ═══════════════════════════════════════════════════════════════
# 18: PRICE RETURN DISTRIBUTION — Heavy Tails, Asymmetry
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("18: Return Distribution Analysis")
print(f"{'='*60}\n")

returns = trades['move'].dropna().values
returns_abs = np.abs(returns)

from scipy.stats import kurtosis, skew, jarque_bera

print(f"  1-minute return statistics:")
print(f"    Mean:     {returns.mean():.7f}")
print(f"    Std:      {returns.std():.7f}")
print(f"    Skew:     {skew(returns):.3f}")
print(f"    Kurtosis: {kurtosis(returns):.3f} (normal=0)")

jb_stat, jb_p = jarque_bera(returns)
print(f"    Jarque-Bera: stat={jb_stat:.1f}, p={jb_p:.4f}")
print(f"    → {'NOT normal (heavy tails / asymmetry)' if jb_p < 0.05 else 'Normal'}")

# If returns are heavy-tailed, extreme moves might be more predictable
extreme_up = returns > np.percentile(returns, 95)
extreme_down = returns < np.percentile(returns, 5)

# After extreme UP return, what happens next?
c_ext = trades.copy()
c_ext['prev_extreme_up'] = (c_ext['move'].shift(1) > np.percentile(returns, 95)).astype(int)
c_ext['prev_extreme_down'] = (c_ext['move'].shift(1) < np.percentile(returns, 5)).astype(int)

for label, col in [('After extreme UP', 'prev_extreme_up'), ('After extreme DOWN', 'prev_extreme_down')]:
    mask = c_ext[col] == 1
    if mask.sum() > 20:
        up_pct = c_ext.loc[mask, 'went_up'].mean()
        follow = (c_ext.loc[mask, 'went_up'] == c_ext.loc[mask, 'prev_went_up']).mean()
        print(f"\n  {label} (top/bottom 5%): P(UP)={up_pct:.1%}, Follow={follow:.1%} ({mask.sum()} trades)")

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(returns * 100000, bins=100, color='cyan', alpha=0.7)
axes[0].set_title('1-Minute Return Distribution (in pips)')
axes[0].set_xlabel('Return (pips)')
axes[1].hist(returns_abs * 100000, bins=100, color='lime', alpha=0.7)
axes[1].set_title('Absolute Return Distribution')
axes[1].set_xlabel('|Return| (pips)')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 19: ORDINAL PATTERNS — Out-of-Sample Validation
# ═══════════════════════════════════════════════════════════════

print("19: Ordinal Pattern Strategy — OOS Validation")
print("=" * 60)

# Split: learn patterns from first 70%, test on last 30%
split = int(len(c_ord) * 0.7)
train_ord = c_ord.iloc[:split]
test_ord = c_ord.iloc[split:]

# Learn optimal direction for each pattern from training data
pattern_bias = train_ord.groupby('prev_ordinal')['went_up'].agg(['mean', 'count']).reset_index()
pattern_bias.columns = ['pattern', 'up_pct', 'count']

# Only use patterns with enough training samples
for min_count in [10, 20, 30, 50]:
    valid_patterns = pattern_bias[pattern_bias['count'] >= min_count]
    pattern_map = dict(zip(valid_patterns['pattern'], valid_patterns['up_pct']))
    
    # Predict on test data
    preds = []
    actuals = []
    for _, row in test_ord.iterrows():
        p = row['prev_ordinal']
        if p in pattern_map:
            pred = 1 if pattern_map[p] > 0.5 else 0
            preds.append(pred)
            actuals.append(int(row['went_up']))
    
    if len(preds) > 100:
        acc = sum(1 for p, a in zip(preds, actuals) if p == a) / len(preds)
        coverage = len(preds) / len(test_ord) * 100
        r = backtest(preds, actuals)
        print(f"  min_count={min_count}: OOS={acc:.1%} ({len(preds)} trades, {coverage:.0f}% coverage)")
        show_result(f"    Ordinal (min_n={min_count})", r)

# ── Use STRONGER bias threshold ──
print(f"\n  With stronger bias thresholds:")
for min_count, min_bias in [(20, 0.55), (20, 0.60), (20, 0.65), (10, 0.60), (10, 0.65)]:
    valid_patterns = pattern_bias[(pattern_bias['count'] >= min_count) & 
                                   (pattern_bias['up_pct'].apply(lambda p: max(p, 1-p)) >= min_bias)]
    pattern_map = dict(zip(valid_patterns['pattern'], valid_patterns['up_pct']))
    
    preds = []
    actuals = []
    for _, row in test_ord.iterrows():
        p = row['prev_ordinal']
        if p in pattern_map:
            pred = 1 if pattern_map[p] > 0.5 else 0
            preds.append(pred)
            actuals.append(int(row['went_up']))
    
    if len(preds) > 30:
        acc = sum(1 for p, a in zip(preds, actuals) if p == a) / len(preds)
        r = backtest(preds, actuals)
        show_result(f"  n≥{min_count}, bias≥{min_bias:.0%} ({len(preds)} trades)", r)

# ═══════════════════════════════════════════════════════════════
# 20: COMBINE BEST SIGNALS — Ordinal + Hour + Fade
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("20: Combined Strategy — All Signals That Showed Life")
print(f"{'='*60}\n")

# Use training data to learn:
# 1. Hourly bias
# 2. Ordinal pattern bias  
# 3. Fade :00 signal

c_final = trades.dropna(subset=[f'c_{s:02d}' for s in [10, 20, 30, 40, 50]]).copy()

# Build ordinal patterns
pattern_cols = ['close', 'c_10', 'c_20', 'c_30', 'c_40', 'c_50']
patterns = c_final[pattern_cols].values
ordinal = np.array([rankdata(row) for row in patterns])
c_final['ordinal'] = [''.join(str(int(x)) for x in row) for row in ordinal]
c_final['prev_ordinal'] = c_final['ordinal'].shift(1)

# Fade signal
c_final['sec0_up'] = (c_final['close'] > c_final['open']).astype(int)
c_final['fade_signal'] = 1 - c_final['sec0_up']

# Previous direction
c_final['prev_dir'] = c_final['went_up'].shift(1)

c_final = c_final.dropna()

split = int(len(c_final) * 0.7)
train_f = c_final.iloc[:split]
test_f = c_final.iloc[split:]

# Learn from training
hour_bias_f = train_f.groupby('hour')['went_up'].mean().to_dict()
ordinal_bias_f = train_f.groupby('prev_ordinal')['went_up'].agg(['mean', 'count'])
ordinal_map_f = {p: r['mean'] for p, r in ordinal_bias_f.iterrows() if r['count'] >= 15}

# Strategy: voting between signals
preds = []
actuals = []
signals_used = []

for _, row in test_f.iterrows():
    h = int(row['hour'])
    ord_p = row['prev_ordinal']
    
    votes = []
    
    # Vote 1: Hourly bias (if strong)
    h_bias = hour_bias_f.get(h, 0.5)
    if abs(h_bias - 0.5) > 0.03:
        votes.append(1 if h_bias > 0.5 else 0)
    
    # Vote 2: Ordinal pattern (if known and strong)
    if ord_p in ordinal_map_f:
        o_bias = ordinal_map_f[ord_p]
        if abs(o_bias - 0.5) > 0.05:
            votes.append(1 if o_bias > 0.5 else 0)
    
    # Vote 3: Fade :00 candle
    votes.append(int(row['fade_signal']))
    
    if len(votes) >= 2:
        # Majority vote
        pred = 1 if sum(votes) > len(votes) / 2 else 0
        preds.append(pred)
        actuals.append(int(row['went_up']))
        signals_used.append(len(votes))

if len(preds) > 100:
    acc = sum(1 for p, a in zip(preds, actuals) if p == a) / len(preds)
    coverage = len(preds) / len(test_f) * 100
    print(f"  Combined voting (≥2 signals):")
    print(f"    OOS Accuracy: {acc:.1%} ({len(preds)} trades, {coverage:.0f}% coverage)")
    r = backtest(preds, actuals)
    show_result("Combined voting", r)

# Also test: trade ALL minutes, use best available signal
preds_all = []
actuals_all = []
for _, row in test_f.iterrows():
    h = int(row['hour'])
    ord_p = row['prev_ordinal']
    
    # Priority: ordinal pattern > hourly bias > fade :00
    if ord_p in ordinal_map_f and abs(ordinal_map_f[ord_p] - 0.5) > 0.10:
        pred = 1 if ordinal_map_f[ord_p] > 0.5 else 0
    elif abs(hour_bias_f.get(h, 0.5) - 0.5) > 0.04:
        pred = 1 if hour_bias_f[h] > 0.5 else 0
    else:
        pred = int(row['fade_signal'])
    
    preds_all.append(pred)
    actuals_all.append(int(row['went_up']))

acc_all = sum(1 for p, a in zip(preds_all, actuals_all) if p == a) / len(preds_all)
print(f"\n  Priority cascade (ordinal > hourly > fade):")
print(f"    OOS Accuracy: {acc_all:.1%} ({len(preds_all)} trades)")
r = backtest(preds_all, actuals_all)
show_result("Priority cascade (all trades)", r)

# Equity curves
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(r['history'], label=f"Priority cascade ({r['win_rate']:.1f}%)", linewidth=2)
# Compare to random baseline
r_rand = backtest([random.randint(0,1) for _ in actuals_all], actuals_all)
ax.plot(r_rand['history'], label=f"Random ({r_rand['win_rate']:.1f}%)", linewidth=1, linestyle='--', color='gray')
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Best Combined Strategy — Out of Sample')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 21: MINUTE BOUNDARY MICRO-STRUCTURE
# ═══════════════════════════════════════════════════════════════
# The live bot enters at :00-:05. What's special about those seconds?
# Maybe the OTC algorithm behaves differently at minute boundaries.

print("21: Minute Boundary Micro-Structure")
print("=" * 60)

# For each minute, look at the last 5 seconds and first 5 seconds
c_bnd = trades.dropna(subset=['c_01', 'c_02', 'c_03', 'c_04', 'c_05']).copy()

# ── 21A: Price jumps at the exact boundary ──
print("\n21A: Second-by-second around the :00 boundary")
print("-" * 60)

# Build price at each second from :55 to :05 (crossing the boundary)
boundary_seconds = [55, 56, 57, 58, 59, 0, 1, 2, 3, 4, 5]

for sec in boundary_seconds:
    col = f'c_{sec:02d}'
    if col in c_bnd.columns:
        if sec >= 55:
            # Previous minute's second
            prev_col = col
            prices = c_bnd[prev_col].shift(1)  # From previous row
        else:
            prices = c_bnd[col]
        
        if sec == 0:
            ref_price = c_bnd['close']  # :00 close
        
# Actually, let's compute the micro-returns at each second
print("  Micro-returns around :00 boundary:")
print("  (Each row shows: avg return from previous second)")
print()

for i, sec in enumerate(range(0, 10)):
    col = f'c_{sec:02d}' if sec > 0 else 'close'
    prev_col = f'c_{sec-1:02d}' if sec > 1 else ('close' if sec == 1 else None)
    
    if prev_col and col in c_bnd.columns and prev_col in c_bnd.columns:
        micro_ret = (c_bnd[col] - c_bnd[prev_col]) * 100000  # In pips
        up_pct = (micro_ret > 0).mean()
        print(f"    :{sec-1:02d}→:{sec:02d}: mean={micro_ret.mean():+.3f} pips, "
              f"std={micro_ret.std():.3f}, UP={up_pct:.1%}")

# ── 21B: Entry price sensitivity ──
print(f"\n21B: How does EXACT entry second affect outcomes?")
print("-" * 60)

# If the bot enters at :00 vs :01 vs :02, the entry price differs slightly
# The RESULT is always at next :00. How much does entry second matter?
for entry_sec in range(0, 6):
    entry_col = f'c_{entry_sec:02d}' if entry_sec > 0 else 'close'
    if entry_col in c_bnd.columns:
        entry_p = c_bnd[entry_col]
        result_p = c_bnd['result_close']  # Next :00 close
        
        # Direction from this entry point
        up = (result_p > entry_p).astype(int)
        
        # How often does THIS entry agree with :00 entry?
        base_up = c_bnd['went_up']
        agree = (up == base_up).mean()
        
        # What % of time does the 1-second difference FLIP the result?
        flip = 1 - agree
        
        print(f"  Entry at :{entry_sec:02d}: agrees with :00 entry {agree:.1%}, "
              f"flips {flip:.1%} of outcomes")

# ── 21C: The CRITICAL question — does the flip create edge? ──
print(f"\n21C: Entry Second Flips — Do They Create Edge?")
print("-" * 60)

# When entry at :02 gives DIFFERENT result than entry at :00,
# which one does the IQ Option platform use?
# The platform uses YOUR entry price (at :02) not the :00 price.
# So if :02 entry flips some trades from loss to win...

for entry_sec in [1, 2, 3]:
    entry_col = f'c_{entry_sec:02d}'
    if entry_col in c_bnd.columns:
        entry_p = c_bnd[entry_col]
        result_p = c_bnd['result_close']
        
        trade_up = (result_p > entry_p).astype(int)
        base_up = c_bnd['went_up']  # :00 entry direction
        
        # Cases where entry at :N gives different result than :00
        flipped = (trade_up != base_up)
        
        # When flipped, what was the :00 result vs :N result?
        if flipped.sum() > 0:
            # For follow-last strategy: we predict based on prev direction
            prev_dir = c_bnd['prev_went_up']
            
            # Accuracy using :00 entry
            acc_00 = (base_up == prev_dir).mean()
            
            # Accuracy using :N entry  
            acc_n = (trade_up == prev_dir).mean()
            
            print(f"\n  Entry :{entry_sec:02d} vs :00:")
            print(f"    Flips: {flipped.sum()} trades ({flipped.mean()*100:.1%})")
            print(f"    Follow-last with :00 entry: {acc_00:.1%}")
            print(f"    Follow-last with :{entry_sec:02d} entry: {acc_n:.1%}")
            print(f"    Difference: {acc_n - acc_00:+.1%}")
            
            # Backtest with :N entry
            r = backtest(prev_dir.values, trade_up.values)
            show_result(f"    Follow-last (:{entry_sec:02d} entry)", r)

# ── 21D: What if the platform uses OPEN not CLOSE? ──
print(f"\n21D: Using OPEN prices instead of CLOSE")
print("-" * 60)

# IQ Option might record entry as the OPEN of the :00 candle
# and expiry as the OPEN of the next :00 candle
c_open = raw[raw['second'] == 0][['open', 'close']].copy()
c_open['next_open'] = c_open['open'].shift(-1)
c_open['up_open'] = (c_open['next_open'] > c_open['open']).astype(int)
c_open['up_close'] = (c_open['close'].shift(-1) > c_open['close']).astype(int)
c_open['prev_up_open'] = c_open['up_open'].shift(1)
c_open = c_open.dropna()

# Open-to-open follow-last
acc_oo = (c_open['up_open'] == c_open['prev_up_open']).mean()
# Close-to-close follow-last  
acc_cc = (c_open['up_close'] == c_open['up_close'].shift(1)).dropna().mean()
# Open-to-open follow close-to-close result
acc_oc = (c_open['up_open'] == c_open['up_close'].shift(1)).dropna().mean()

print(f"  Follow-last accuracy:")
print(f"    Close→Close (our standard): {acc_cc:.1%}")
print(f"    Open→Open:                  {acc_oo:.1%}")
print(f"    Use Close result → predict Open: {acc_oc:.1%}")

# KEY: what if open and close differ at :00?
c_open['oc_diff'] = (c_open['close'] - c_open['open']).abs() * 100000
print(f"\n  :00 candle Open vs Close difference:")
print(f"    Mean: {c_open['oc_diff'].mean():.2f} pips")
print(f"    Median: {c_open['oc_diff'].median():.2f} pips")
print(f"    % where O ≠ C: {(c_open['oc_diff'] > 0).mean()*100:.1f}%")

# When O ≠ C at :00, does it matter which one the platform uses?
diff_mask = c_open['oc_diff'] > 0
if diff_mask.sum() > 100:
    # Does the open-to-open direction differ from close-to-close?
    disagree = (c_open.loc[diff_mask, 'up_open'] != c_open.loc[diff_mask, 'up_close']).mean()
    print(f"    When O≠C: open direction ≠ close direction {disagree:.1%} of the time")
    print(f"    These are {int(disagree * diff_mask.sum())} trades where the entry price definition matters")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 22: FIXED DIRECTIONAL BIAS BY SECOND
# ═══════════════════════════════════════════════════════════════
# For each second 0-59: if you enter at that second,
# what % of the time is the price HIGHER at the end of the minute?
# This tests a FIXED directional bias at each second.

print("22: Fixed Directional Bias by Entry Second")
print("=" * 60)

# For each minute, we have the price at every second (c_00 to c_59)
# Result = next :00 close (result_close)

c_sec = trades.copy()
result_price = c_sec['result_close'].values

results_by_sec = []

for sec in range(0, 60):
    col = f'c_{sec:02d}' if sec > 0 else 'close'
    if col not in c_sec.columns:
        continue
    
    valid = c_sec.dropna(subset=[col])
    entry_price = valid[col].values
    result = valid['result_close'].values
    
    # P(price goes UP from this second to end of minute)
    up = (result > entry_price).astype(int)
    p_up = up.mean()
    n = len(up)
    
    # Statistical significance
    z = (p_up - 0.5) / np.sqrt(0.25 / n) if n > 0 else 0
    
    results_by_sec.append({
        'second': sec,
        'p_up': p_up,
        'p_down': 1 - p_up,
        'best': max(p_up, 1 - p_up),
        'direction': 'UP' if p_up > 0.5 else 'DOWN',
        'z': abs(z),
        'significant': abs(z) > 1.96,
        'n': n,
    })

rdf = pd.DataFrame(results_by_sec)

# Print results
print(f"\n  {'Sec':>4} {'P(UP)':>7} {'Best':>6} {'Dir':>5} {'z':>6} {'Sig':>5} {'N':>6}")
print(f"  {'-'*45}")
for _, row in rdf.iterrows():
    marker = " ★★★" if row['z'] > 2.58 else (" ★★" if row['z'] > 1.96 else (" ★" if row['best'] > 0.52 else ""))
    print(f"  {int(row['second']):>4} {row['p_up']:>6.1%} {row['best']:>5.1%} {row['direction']:>5} "
          f"{row['z']:>5.2f} {'YES' if row['significant'] else 'no':>5} {int(row['n']):>6}{marker}")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# P(UP) by second
colors = ['lime' if p > 0.52 else ('red' if p < 0.48 else 'gray') for p in rdf['p_up']]
axes[0].bar(rdf['second'], rdf['p_up'], color=colors, alpha=0.7)
axes[0].axhline(y=0.5, color='white', linestyle='--', alpha=0.5)
axes[0].axhline(y=0.5405, color='red', linestyle=':', label='Break-even UP (54%)', alpha=0.7)
axes[0].axhline(y=0.4595, color='red', linestyle=':', label='Break-even DOWN (46%)', alpha=0.7)
axes[0].set_title('P(price goes UP from second S to end of minute)')
axes[0].set_xlabel('Entry second')
axes[0].set_ylabel('P(UP)')
axes[0].legend()
axes[0].set_ylim(0.44, 0.56)

# Z-score significance
colors_z = ['lime' if z > 1.96 else 'gray' for z in rdf['z']]
axes[1].bar(rdf['second'], rdf['z'], color=colors_z, alpha=0.7)
axes[1].axhline(y=1.96, color='red', linestyle='--', label='p=0.05 threshold')
axes[1].axhline(y=2.58, color='yellow', linestyle='--', label='p=0.01 threshold')
axes[1].set_title('Statistical Significance (z-score)')
axes[1].set_xlabel('Entry second')
axes[1].set_ylabel('|z-score|')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Backtest the best seconds ──
print(f"\n--- Backtest: Bet the bias direction at each second ---\n")

sig_seconds = rdf[rdf['significant']].sort_values('best', ascending=False)
if len(sig_seconds) > 0:
    print(f"  Statistically significant seconds:")
    for _, row in sig_seconds.iterrows():
        sec = int(row['second'])
        col = f'c_{sec:02d}' if sec > 0 else 'close'
        valid = c_sec.dropna(subset=[col])
        entry_p = valid[col].values
        result_p = valid['result_close'].values
        trade_up = (result_p > entry_p).astype(int)
        
        # Always bet the bias direction
        pred = np.ones(len(trade_up)) if row['direction'] == 'UP' else np.zeros(len(trade_up))
        r = backtest(pred, trade_up)
        show_result(f"  Always {row['direction']} at :{sec:02d}", r)
else:
    print("  No statistically significant seconds found")

# ── Combined: only trade the most biased seconds ──
print(f"\n--- Combined: Trade multiple biased seconds ---\n")

# Strategy: for each minute, enter at the MOST biased second
best_secs = rdf[rdf['best'] > 0.52].sort_values('best', ascending=False)
if len(best_secs) > 0:
    print(f"  Best seconds (>52% bias): {list(best_secs['second'].values)}")
    
    preds = []
    actuals = []
    for _, trade_row in c_sec.dropna(subset=['result_close']).iterrows():
        for _, sec_row in best_secs.iterrows():
            sec = int(sec_row['second'])
            col = f'c_{sec:02d}' if sec > 0 else 'close'
            if col in trade_row.index and not pd.isna(trade_row[col]):
                entry_p = trade_row[col]
                result_p = trade_row['result_close']
                actual_up = int(result_p > entry_p)
                pred = 1 if sec_row['direction'] == 'UP' else 0
                preds.append(pred)
                actuals.append(actual_up)
                break  # Only one trade per minute
    
    if len(preds) > 100:
        acc = sum(1 for p, a in zip(preds, actuals) if p == a) / len(preds)
        r = backtest(preds, actuals)
        show_result(f"Best biased second per minute", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 23: COMBINE SECOND BIAS + FOLLOW-LAST
# ═══════════════════════════════════════════════════════════════
# The price rises during the minute then drops at :00 boundary.
# Can we combine this structural bias with follow-last?

print("23: Second Bias + Follow-Last Combinations")
print("=" * 60)

c_combo = trades.copy()

# For each entry second, test follow-last accuracy
# AND test: follow-last only when it agrees with the DOWN bias
print("\n  Follow-last accuracy by entry second:")
print(f"  {'Entry':>6} {'FollowLast':>10} {'AlwaysDOWN':>10} {'Agree(DOWN)':>12} {'AgreeN':>7}")
print(f"  {'-'*50}")

combo_results = []

for sec in [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 58, 59]:
    col = f'c_{sec:02d}' if sec > 0 else 'close'
    if col not in c_combo.columns:
        continue
    
    valid = c_combo.dropna(subset=[col]).copy()
    entry_p = valid[col].values
    result_p = valid['result_close'].values
    trade_up = (result_p > entry_p).astype(int)
    
    # Follow-last: use previous :00→:00 direction to predict :sec→:00
    prev_dir = valid['prev_went_up'].values
    follow_acc = (trade_up == prev_dir).mean()
    
    # Always DOWN
    down_acc = (trade_up == 0).mean()
    
    # Agree: follow-last says DOWN AND bias says DOWN → bet DOWN
    follow_says_down = prev_dir == 0
    agree_mask = follow_says_down  # Both say DOWN
    if agree_mask.sum() > 100:
        agree_acc = (trade_up[agree_mask] == 0).mean()
        agree_n = agree_mask.sum()
    else:
        agree_acc = 0
        agree_n = 0
    
    marker = " ★" if agree_acc > 0.54 or down_acc > 0.54 else ""
    print(f"  :{sec:>4} {follow_acc:>9.1%} {down_acc:>9.1%} {agree_acc:>11.1%} {agree_n:>7}{marker}")
    
    combo_results.append({
        'sec': sec, 'follow': follow_acc, 'down': down_acc,
        'agree': agree_acc, 'agree_n': agree_n
    })

# ── Full strategy simulations ──
print(f"\n{'='*60}")
print("Full Strategy Simulations with Martingale")
print(f"{'='*60}\n")

# Strategy A: Enter at :55, always bet DOWN
print("A) Enter at :55, always DOWN:")
valid = c_combo.dropna(subset=['c_55']).copy()
trade_up = (valid['result_close'] > valid['c_55']).astype(int)
preds = np.zeros(len(trade_up))
r = backtest(preds, trade_up.values)
show_result("  :55 always DOWN", r)

# Strategy B: Enter at :55, follow-last
print("\nB) Enter at :55, follow-last:")
preds = valid['prev_went_up'].values
trade_up_vals = trade_up.values
r = backtest(preds, trade_up_vals)
show_result("  :55 follow-last", r)

# Strategy C: Enter at :55, follow-last ONLY when it says DOWN, skip when UP
print("\nC) Enter at :55, follow-last only when DOWN (skip UP):")
down_mask = valid['prev_went_up'] == 0
preds_c = np.zeros(int(down_mask.sum()))  # All DOWN
actuals_c = trade_up.values[down_mask]
skip_pct = (1 - down_mask.mean()) * 100
r = backtest(preds_c, actuals_c)
show_result(f"  :55 follow-DOWN only (skip {skip_pct:.0f}%)", r)

# Strategy D: Enter at :00, use follow-last, but add DOWN bias
# If follow-last says DOWN → bet DOWN (bias agrees)
# If follow-last says UP → still bet UP (no bias help)
# This is just follow-last... same as before

# Strategy E: Enter at :59, always DOWN (highest bias)
print("\nE) Enter at :59, always DOWN:")
valid59 = c_combo.dropna(subset=['c_59']).copy()
trade_up_59 = (valid59['result_close'] > valid59['c_59']).astype(int)
preds = np.zeros(len(trade_up_59))
r = backtest(preds, trade_up_59.values)
show_result("  :59 always DOWN", r)

# Strategy F: Simulate actual bot timing
# Bot sees result at :00, decides direction, clicks at :02-:03
# The DOWN bias at :02 is only 50.3%, not enough
# BUT: what if the bot could enter at :55 instead?
print("\nF) Enter at :50, always DOWN:")
valid50 = c_combo.dropna(subset=['c_50']).copy()
trade_up_50 = (valid50['result_close'] > valid50['c_50']).astype(int)
preds = np.zeros(len(trade_up_50))
r = backtest(preds, trade_up_50.values)
show_result("  :50 always DOWN", r)

# ── Strategy G: LATE ENTRY with follow-last combining ──
print(f"\n{'='*60}")
print("G) Late Entry Strategy — Enter :55, smart direction")
print(f"{'='*60}\n")

# At :55, we know:
# 1. The current minute's :00→:55 direction (did price go up in first 55s?)
# 2. The previous minute's result (:00→:00)
# 3. The structural DOWN bias at :55

valid55 = c_combo.dropna(subset=['c_55']).copy()

# Signal: :00→:55 direction (CAUSAL — known at :55)
valid55['curr_min_up'] = (valid55['c_55'] > valid55['close']).astype(int)

# Trade: :55→next:00
valid55['trade_up'] = (valid55['result_close'] > valid55['c_55']).astype(int)

# Test combinations
strategies_g = []

# G1: Always DOWN
pred = np.zeros(len(valid55))
acc = (valid55['trade_up'] == 0).mean()
r = backtest(pred, valid55['trade_up'].values)
r['label'] = 'Always DOWN'
strategies_g.append((acc, r))

# G2: Fade current minute (if minute went UP, bet DOWN for remaining 5s)
pred = 1 - valid55['curr_min_up'].values
acc = (valid55['trade_up'].values == pred).mean()
r = backtest(pred, valid55['trade_up'].values)
r['label'] = 'Fade current minute'
strategies_g.append((acc, r))

# G3: Follow current minute direction
pred = valid55['curr_min_up'].values
acc = (valid55['trade_up'].values == pred).mean()
r = backtest(pred, valid55['trade_up'].values)
r['label'] = 'Follow current minute'
strategies_g.append((acc, r))

# G4: DOWN when current minute went UP (fade + bias agree), UP when went DOWN
pred = 1 - valid55['curr_min_up'].values  # Same as fade
r['label'] = 'Fade + DOWN bias'

# G5: DOWN only when current minute went UP (skip otherwise)
up_mask = valid55['curr_min_up'] == 1
if up_mask.sum() > 100:
    pred = np.zeros(int(up_mask.sum()))
    actuals = valid55.loc[up_mask, 'trade_up'].values
    acc = (actuals == 0).mean()
    r = backtest(pred, actuals)
    r['label'] = f'DOWN when min UP (skip {(1-up_mask.mean())*100:.0f}%)'
    strategies_g.append((acc, r))

# G6: DOWN only when current minute went UP AND follow-last says DOWN
both_down = (valid55['curr_min_up'] == 1) & (valid55['prev_went_up'] == 0)
if both_down.sum() > 50:
    pred = np.zeros(int(both_down.sum()))
    actuals = valid55.loc[both_down, 'trade_up'].values
    acc = (actuals == 0).mean()
    r = backtest(pred, actuals)
    skip_pct = (1 - both_down.mean()) * 100
    r['label'] = f'DOWN: minUP+followDOWN (skip {skip_pct:.0f}%)'
    strategies_g.append((acc, r))

strategies_g.sort(key=lambda x: x[0], reverse=True)
for acc, r in strategies_g:
    show_result(f"  {r['label']}", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 24: REAL TRADE WINDOWS — IQ Option Expiry Rules
# ═══════════════════════════════════════════════════════════════
# IQ Option turbo = 60 second minimum
# If you enter at :55, option expires at :00 of NEXT minute (5s? or 65s?)
# If you enter at :59, option expires at :00 two minutes later (61s)
# Need to test ACTUAL trade windows

print("24: Real IQ Option Trade Windows")
print("=" * 60)

# IQ Option rule: trade expires at the NEXT minute boundary
# that is at least 30 seconds away (approximately)
# So entering at :55 → expires at next :00 = 5 seconds (TOO SHORT, rejected)
# Entering at :55 → expires at :00 + 1 minute = 65 seconds
# Entering at :30 → expires at next :00 = 30 seconds (might be accepted)
# Entering at :00 → expires at next :00 = 60 seconds

# Let's test with REALISTIC windows
print("\n  Testing realistic IQ Option windows:\n")

c_real = trades.copy()

for entry_sec, expiry_offset in [
    (0, 1),    # :00 → next :00 (60s) — our standard
    (55, 2),   # :55 → :00 two minutes later (65s)
    (50, 2),   # :50 → :00 two minutes later (70s)
    (45, 2),   # :45 → :00 two minutes later (75s)
    (30, 1),   # :30 → next :00 (30s) — might be too short
    (30, 2),   # :30 → :00 two minutes later (90s)
    (59, 2),   # :59 → :00 two minutes later (61s)
]:
    entry_col = f'c_{entry_sec:02d}' if entry_sec > 0 else 'close'
    if entry_col not in c_real.columns:
        continue
    
    # Expiry price: :00 close `expiry_offset` minutes later
    expiry_price = c_real['close'].shift(-expiry_offset)
    
    valid = c_real.dropna(subset=[entry_col])
    valid_mask = expiry_price.notna()
    valid = valid[valid_mask]
    expiry = expiry_price[valid_mask]
    
    entry_p = valid[entry_col].values
    result_p = expiry.reindex(valid.index).values
    
    trade_up = (result_p > entry_p).astype(int)
    
    # P(DOWN)
    p_down = (trade_up == 0).mean()
    window_secs = (60 - entry_sec) + (expiry_offset - 1) * 60
    
    # Follow-last (using previous :00→:00 result)
    prev_dir = valid['prev_went_up'].values
    follow_acc = (trade_up == prev_dir).mean()
    
    # Always DOWN
    down_acc = p_down
    
    # Agree: follow-DOWN + always-DOWN
    follow_down = prev_dir == 0
    agree_acc = (trade_up[follow_down] == 0).mean() if follow_down.sum() > 100 else 0
    
    marker = " ★" if down_acc > 0.54 or agree_acc > 0.54 else ""
    print(f"  :{entry_sec:02d}→:00+{expiry_offset}min ({window_secs}s): "
          f"DOWN={down_acc:.1%} Follow={follow_acc:.1%} Agree={agree_acc:.1%}{marker}")

# ── Full backtest of the most promising ──
print(f"\n{'='*60}")
print("Backtests — Real IQ Option Windows")
print(f"{'='*60}\n")

for entry_sec, expiry_offset, label in [
    (0, 1, ":00 entry, 60s trade"),
    (55, 2, ":55 entry, 65s trade"),
    (59, 2, ":59 entry, 61s trade"),
    (50, 2, ":50 entry, 70s trade"),
]:
    entry_col = f'c_{entry_sec:02d}' if entry_sec > 0 else 'close'
    if entry_col not in c_real.columns:
        continue
    
    expiry_price = c_real['close'].shift(-expiry_offset)
    valid = c_real.dropna(subset=[entry_col])
    valid = valid[expiry_price.notna()]
    expiry = expiry_price.reindex(valid.index)
    
    trade_up = (expiry.values > valid[entry_col].values).astype(int)
    
    # Always DOWN
    preds = np.zeros(len(trade_up))
    r = backtest(preds, trade_up)
    show_result(f"{label} always DOWN", r)
    
    # Follow-last says DOWN only
    prev_dir = valid['prev_went_up'].values
    down_mask = prev_dir == 0
    if down_mask.sum() > 100:
        preds_d = np.zeros(int(down_mask.sum()))
        actuals_d = trade_up[down_mask]
        r = backtest(preds_d, actuals_d)
        show_result(f"{label} follow+DOWN agree", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 25: SUB-MINUTE TIMESCALES — Where Does Predictability Live?
# ═══════════════════════════════════════════════════════════════
# We tested 1-min direction: 50%. But what about 5s, 10s, 15s, 30s?
# The pattern might exist at a different timescale.

print("25: Follow-Last at EVERY Timescale (1s to 5min)")
print("=" * 60)

# Test follow-last (non-overlapping) at different timescales
# using the RAW 1-second data

timescale_results = []

for window_sec in [1, 2, 3, 5, 10, 15, 20, 30, 45, 60, 90, 120, 180, 300]:
    # Sample every window_sec seconds from the raw data
    step = window_sec  # rows in 1-second data
    
    prices = raw['close'].values
    
    # Build non-overlapping windows
    directions = []
    for i in range(0, len(prices) - step, step):
        if i + step < len(prices):
            went_up = int(prices[i + step] > prices[i])
            directions.append(went_up)
    
    if len(directions) < 100:
        continue
    
    # Follow-last accuracy (non-overlapping)
    dirs = np.array(directions)
    matches = sum(1 for i in range(1, len(dirs)) if dirs[i] == dirs[i-1])
    total = len(dirs) - 1
    follow_acc = matches / total if total > 0 else 0
    
    # Always UP accuracy
    up_pct = dirs.mean()
    
    timescale_results.append({
        'window': window_sec,
        'follow_acc': follow_acc,
        'up_pct': up_pct,
        'n': total,
    })

tdf = pd.DataFrame(timescale_results)

print(f"\n  {'Window':>8} {'Follow%':>8} {'UP%':>6} {'Trades':>7} {'Profitable':>10}")
print(f"  {'-'*45}")
for _, row in tdf.iterrows():
    w = row['window']
    label = f"{w}s" if w < 60 else f"{w//60}m{w%60}s" if w % 60 else f"{w//60}min"
    profitable = "YES ★" if row['follow_acc'] > 0.5405 else "no"
    print(f"  {label:>8} {row['follow_acc']:>7.1%} {row['up_pct']:>5.1%} {int(row['n']):>7} {profitable:>10}")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(tdf['window'], tdf['follow_acc'], 'o-', color='cyan', linewidth=2, markersize=8)
ax.axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
ax.axhline(y=0.5405, color='red', linestyle='--', label='Break-even (54%)')
ax.set_title('Follow-Last Accuracy by Timescale (Non-Overlapping)')
ax.set_xlabel('Window size (seconds)')
ax.set_ylabel('Follow-last accuracy')
ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 26: PRICE LEVEL REGRESSION — Is There Mean Reversion?
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("26: Price Level Auto-Regression")
print(f"{'='*60}\n")

# Fit AR(1) model: price[t] = a + b * price[t-1] + noise
# If b < 1: mean reversion
# If b > 1: momentum
# If b = 1: random walk

from scipy.stats import linregress

prices_1min = trades['close'].values

# AR(1) at different timescales
print("  AR(1) coefficient by timescale:")
print(f"  {'Scale':>8} {'β':>8} {'R²':>8} {'Interpretation':>20}")
print(f"  {'-'*50}")

for lag in [1, 2, 5, 10, 30, 60]:
    if lag < len(prices_1min):
        y = prices_1min[lag:]
        x = prices_1min[:-lag]
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        
        if slope > 1.001:
            interp = "MOMENTUM"
        elif slope < 0.999:
            interp = "MEAN REVERSION"
        else:
            interp = "Random walk"
        
        label = f"{lag}min" if lag >= 1 else f"{lag*60}s"
        print(f"  {label:>8} {slope:>7.5f} {r_value**2:>7.4f} {interp:>20}")

# ═══════════════════════════════════════════════════════════════
# 27: CONDITIONAL ANALYSIS — The Interaction Effect
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("27: What If the Edge Only Exists in Specific Conditions?")
print(f"{'='*60}\n")

# Maybe follow-last works but ONLY during certain market states
# Test follow-last conditioned on: volatility regime + hour + move size

c_cond = trades.copy()
c_cond['prev_move_abs'] = c_cond['move_abs'].shift(1)
c_cond['vol_20'] = c_cond['move_abs'].rolling(20).std()
c_cond['prev_vol'] = c_cond['vol_20'].shift(1)
c_cond = c_cond.dropna()

# Split into volatility regimes
c_cond['vol_regime'] = pd.qcut(c_cond['prev_vol'], q=3, labels=['Low Vol', 'Med Vol', 'High Vol'])

# Test follow-last in each regime × hour combination
print("  Follow-last by volatility regime:")
for regime in ['Low Vol', 'Med Vol', 'High Vol']:
    mask = c_cond['vol_regime'] == regime
    sub = c_cond[mask]
    acc = (sub['went_up'] == sub['prev_went_up']).mean()
    n = len(sub)
    marker = " ★" if acc > 0.52 else ""
    print(f"    {regime:>8}: {acc:.1%} ({n} trades){marker}")

# Test: follow-last in low volatility + specific hours
print(f"\n  Follow-last by regime × time-of-day:")
for regime in ['Low Vol', 'Med Vol', 'High Vol']:
    for period, hours in [('Night 0-6', range(0,6)), ('Day 6-18', range(6,18)), ('Eve 18-24', range(18,24))]:
        mask = (c_cond['vol_regime'] == regime) & (c_cond['hour'].isin(hours))
        sub = c_cond[mask]
        if len(sub) > 200:
            acc = (sub['went_up'] == sub['prev_went_up']).mean()
            marker = " ★" if acc > 0.53 else ""
            print(f"    {regime:>8} + {period:<10}: {acc:.1%} ({len(sub)} trades){marker}")

# Test: does the price at specific FRACTIONAL levels predict?
# e.g., price ending in .000XX where XX is 00-99
print(f"\n  Follow-last by price's last 2 digits (pip position):")
c_cond['last_2_digits'] = (np.round(c_cond['close'] * 10000) % 10).astype(int)  # 4th decimal
for digit in range(10):
    mask = c_cond['last_2_digits'] == digit
    sub = c_cond[mask]
    if len(sub) > 200:
        acc = (sub['went_up'] == sub['prev_went_up']).mean()
        up_pct = sub['went_up'].mean()
        marker = " ★" if acc > 0.52 or abs(up_pct - 0.5) > 0.03 else ""
        print(f"    4th decimal = {digit}: Follow={acc:.1%}, UP={up_pct:.1%} ({len(sub)}){marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 28: MOVING AVERAGE CROSSOVER — The Classic Trend Signal
# ═══════════════════════════════════════════════════════════════
# The IQ Option chart shows clear trends with MAs.
# When short MA > long MA → trend is UP → bet UP
# This is DIFFERENT from what we tested before (price vs MA).

print("28: Moving Average Crossover Strategy")
print("=" * 60)

c_ma = trades.copy()

# Compute MAs on the :00 close price (all CAUSAL — known at entry time)
for window in [3, 5, 7, 10, 15, 20, 30, 50]:
    c_ma[f'ma_{window}'] = c_ma['close'].rolling(window).mean()

# Also compute MA slope (is the MA going up or down?)
for window in [5, 10, 20]:
    c_ma[f'ma_{window}_slope'] = c_ma[f'ma_{window}'] - c_ma[f'ma_{window}'].shift(1)

c_ma = c_ma.dropna()

# ── Test 1: MA Crossover signals ──
print("\n  MA Crossover: Short MA > Long MA → bet UP")
print(f"  {'Crossover':<20} {'Accuracy':>8} {'Trades':>7}")
print(f"  {'-'*40}")

for short_w, long_w in [(3,7), (3,10), (5,10), (5,15), (5,20), (7,20), (10,20), (10,30), (10,50), (20,50)]:
    short_col = f'ma_{short_w}'
    long_col = f'ma_{long_w}'
    if short_col in c_ma.columns and long_col in c_ma.columns:
        # Signal: short MA > long MA → UP
        signal_up = (c_ma[short_col] > c_ma[long_col]).astype(int)
        acc = (c_ma['went_up'] == signal_up).mean()
        marker = " ★" if acc > 0.52 else ""
        print(f"  MA{short_w}/MA{long_w:<14} {acc:>7.1%} {len(c_ma):>7}{marker}")

# ── Test 2: MA Slope — is the trend accelerating? ──
print(f"\n  MA Slope: MA going UP → bet UP")
print(f"  {'Signal':<20} {'Accuracy':>8}")
print(f"  {'-'*35}")

for window in [5, 10, 20]:
    slope_col = f'ma_{window}_slope'
    if slope_col in c_ma.columns:
        signal_up = (c_ma[slope_col] > 0).astype(int)
        acc = (c_ma['went_up'] == signal_up).mean()
        marker = " ★" if acc > 0.52 else ""
        print(f"  MA{window} slope UP     {acc:>7.1%}{marker}")

# ── Test 3: Price relative to MA — FOLLOW the trend, don't fade ──
print(f"\n  Price > MA → bet UP (trend following, not mean reversion)")
for window in [5, 10, 20, 50]:
    ma_col = f'ma_{window}'
    if ma_col in c_ma.columns:
        signal_up = (c_ma['close'] > c_ma[ma_col]).astype(int)
        acc = (c_ma['went_up'] == signal_up).mean()
        marker = " ★" if acc > 0.52 else ""
        print(f"  Price > MA{window:<8} {acc:>7.1%}{marker}")

# ── Test 4: Triple MA — short > medium > long = strong uptrend ──
print(f"\n  Triple MA alignment:")
for s, m, l in [(3, 10, 30), (5, 15, 50), (5, 10, 20), (3, 7, 20)]:
    s_col, m_col, l_col = f'ma_{s}', f'ma_{m}', f'ma_{l}'
    if all(c in c_ma.columns for c in [s_col, m_col, l_col]):
        bullish = (c_ma[s_col] > c_ma[m_col]) & (c_ma[m_col] > c_ma[l_col])
        bearish = (c_ma[s_col] < c_ma[m_col]) & (c_ma[m_col] < c_ma[l_col])
        neutral = ~bullish & ~bearish
        
        if bullish.sum() > 100 and bearish.sum() > 100:
            bull_acc = c_ma.loc[bullish, 'went_up'].mean()
            bear_acc = 1 - c_ma.loc[bearish, 'went_up'].mean()
            neut_up = c_ma.loc[neutral, 'went_up'].mean()
            
            # Strategy: bet UP when bullish, DOWN when bearish, skip neutral
            aligned = bullish | bearish
            pred = bullish.astype(int)
            acc = (c_ma.loc[aligned, 'went_up'] == pred[aligned]).mean()
            skip_pct = neutral.mean() * 100
            
            marker = " ★" if acc > 0.52 else ""
            print(f"  MA{s}/{m}/{l}: Bull→UP={bull_acc:.1%}({bullish.sum()}) "
                  f"Bear→DOWN={bear_acc:.1%}({bearish.sum()}) "
                  f"Aligned={acc:.1%} (skip {skip_pct:.0f}%){marker}")

# ── Backtest best MA strategies ──
print(f"\n{'='*60}")
print("Backtests — MA Strategies")
print(f"{'='*60}\n")

# Best crossover
for short_w, long_w in [(3,10), (5,20), (10,30)]:
    short_col = f'ma_{short_w}'
    long_col = f'ma_{long_w}'
    signal_up = (c_ma[short_col] > c_ma[long_col]).astype(int)
    r = backtest(signal_up.values, c_ma['went_up'].values)
    show_result(f"MA{short_w}/MA{long_w} crossover", r)

# Best slope
for window in [5, 10]:
    slope_col = f'ma_{window}_slope'
    signal_up = (c_ma[slope_col] > 0).astype(int)
    r = backtest(signal_up.values, c_ma['went_up'].values)
    show_result(f"MA{window} slope direction", r)

# Triple MA — only trade when aligned
for s, m, l in [(5, 10, 20), (3, 10, 30)]:
    s_col, m_col, l_col = f'ma_{s}', f'ma_{m}', f'ma_{l}'
    if all(c in c_ma.columns for c in [s_col, m_col, l_col]):
        bullish = (c_ma[s_col] > c_ma[m_col]) & (c_ma[m_col] > c_ma[l_col])
        bearish = (c_ma[s_col] < c_ma[m_col]) & (c_ma[m_col] < c_ma[l_col])
        aligned = bullish | bearish
        pred = bullish[aligned].astype(int).values
        actual = c_ma.loc[aligned, 'went_up'].values
        r = backtest(pred, actual)
        skip_pct = (~aligned).mean() * 100
        show_result(f"Triple MA{s}/{m}/{l} (skip {skip_pct:.0f}%)", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 29: TECHNICAL INDICATORS — RSI, Bollinger, MACD, Distance
# ═══════════════════════════════════════════════════════════════

print("29: Classic Technical Indicators")
print("=" * 60)

c_ti = trades.copy()

# ── RSI (Relative Strength Index) ──
def compute_rsi(prices, period=14):
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

for period in [5, 10, 14, 20]:
    c_ti[f'rsi_{period}'] = compute_rsi(c_ti['close'], period)

# ── Bollinger Bands ──
for period in [10, 20]:
    ma = c_ti['close'].rolling(period).mean()
    std = c_ti['close'].rolling(period).std()
    c_ti[f'bb_upper_{period}'] = ma + 2 * std
    c_ti[f'bb_lower_{period}'] = ma - 2 * std
    c_ti[f'bb_pct_{period}'] = (c_ti['close'] - c_ti[f'bb_lower_{period}']) / (c_ti[f'bb_upper_{period}'] - c_ti[f'bb_lower_{period}'])

# ── MACD ──
ema12 = c_ti['close'].ewm(span=12).mean()
ema26 = c_ti['close'].ewm(span=26).mean()
c_ti['macd'] = ema12 - ema26
c_ti['macd_signal'] = c_ti['macd'].ewm(span=9).mean()
c_ti['macd_hist'] = c_ti['macd'] - c_ti['macd_signal']

# ── MA Distance (how far price is from MA, normalized) ──
for w in [10, 20, 50]:
    ma = c_ti['close'].rolling(w).mean()
    std = c_ti['close'].rolling(w).std()
    c_ti[f'ma_dist_{w}'] = (c_ti['close'] - ma) / std  # Z-score

c_ti = c_ti.dropna()

# ── Test each indicator ──
print(f"\n  {'Signal':<40} {'Accuracy':>8} {'N':>6}")
print(f"  {'-'*58}")

signals = []

# RSI signals
for period in [5, 10, 14, 20]:
    col = f'rsi_{period}'
    # RSI > 50 → UP (momentum)
    sig = (c_ti[col] > 50).astype(int)
    acc = (c_ti['went_up'] == sig).mean()
    signals.append((f'RSI{period} > 50 → UP', acc))
    
    # RSI > 70 → overbought → DOWN (only when overbought)
    overbought = c_ti[col] > 70
    if overbought.sum() > 50:
        acc_ob = 1 - c_ti.loc[overbought, 'went_up'].mean()
        signals.append((f'RSI{period} > 70 → DOWN ({overbought.sum()})', acc_ob))
    
    oversold = c_ti[col] < 30
    if oversold.sum() > 50:
        acc_os = c_ti.loc[oversold, 'went_up'].mean()
        signals.append((f'RSI{period} < 30 → UP ({oversold.sum()})', acc_os))

# Bollinger Band signals
for period in [10, 20]:
    # Price above upper band → overbought → DOWN
    above = c_ti['close'] > c_ti[f'bb_upper_{period}']
    below = c_ti['close'] < c_ti[f'bb_lower_{period}']
    if above.sum() > 50:
        acc = 1 - c_ti.loc[above, 'went_up'].mean()
        signals.append((f'BB{period} above upper → DOWN ({above.sum()})', acc))
    if below.sum() > 50:
        acc = c_ti.loc[below, 'went_up'].mean()
        signals.append((f'BB{period} below lower → UP ({below.sum()})', acc))
    
    # BB %B as trend signal
    sig = (c_ti[f'bb_pct_{period}'] > 0.5).astype(int)
    acc = (c_ti['went_up'] == sig).mean()
    signals.append((f'BB{period} %B > 0.5 → UP', acc))

# MACD signals
sig = (c_ti['macd'] > c_ti['macd_signal']).astype(int)
acc = (c_ti['went_up'] == sig).mean()
signals.append(('MACD > Signal → UP', acc))

sig = (c_ti['macd'] > 0).astype(int)
acc = (c_ti['went_up'] == sig).mean()
signals.append(('MACD > 0 → UP', acc))

sig = (c_ti['macd_hist'] > 0).astype(int)
acc = (c_ti['went_up'] == sig).mean()
signals.append(('MACD histogram > 0 → UP', acc))

# MA Distance — bet when price is FAR from MA (strong trend)
for w in [10, 20, 50]:
    col = f'ma_dist_{w}'
    # Far above MA (z > 1) → strong uptrend → UP
    far_above = c_ti[col] > 1
    far_below = c_ti[col] < -1
    if far_above.sum() > 50:
        acc = c_ti.loc[far_above, 'went_up'].mean()
        signals.append((f'MA{w} z>1 (strong up) → UP ({far_above.sum()})', acc))
    if far_below.sum() > 50:
        acc = 1 - c_ti.loc[far_below, 'went_up'].mean()
        signals.append((f'MA{w} z<-1 (strong dn) → DOWN ({far_below.sum()})', acc))

# Sort by accuracy
signals.sort(key=lambda x: x[1], reverse=True)
for name, acc in signals:
    marker = " ★" if acc > 0.53 else ""
    print(f"  {name:<40} {acc:>7.1%}{marker}")

# ── Backtest top signals ──
print(f"\n{'='*60}")
print("Backtests — Best Technical Indicators")
print(f"{'='*60}\n")

# MACD crossover
sig = (c_ti['macd'] > c_ti['macd_signal']).astype(int)
r = backtest(sig.values, c_ti['went_up'].values)
show_result("MACD > Signal → UP", r)

# RSI14 > 50
sig = (c_ti['rsi_14'] > 50).astype(int)
r = backtest(sig.values, c_ti['went_up'].values)
show_result("RSI14 > 50 → UP", r)

# MA distance z>1 / z<-1
col = 'ma_dist_20'
far = (c_ti[col].abs() > 1)
if far.sum() > 200:
    pred = (c_ti.loc[far, col] > 0).astype(int).values
    actual = c_ti.loc[far, 'went_up'].values
    r = backtest(pred, actual)
    show_result(f"MA20 |z|>1 trend follow (skip {(~far).mean()*100:.0f}%)", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 30: VISUAL PROOF — Random Walk vs OTC Chart
# ═══════════════════════════════════════════════════════════════
# Does the OTC chart look different from a pure random walk?

print("30: Can You Tell Which Is Real?")
print("=" * 60)

# Take a 2-hour chunk of real OTC data
real_prices = trades['close'].values[:120]  # 120 minutes = 2 hours

# Generate a random walk with same properties
np.random.seed(7)  # Pick a seed that looks "trendy"
steps = np.random.choice([-1, 1], size=120) * np.std(np.diff(real_prices))
random_prices = real_prices[0] + np.cumsum(steps)

# Add MAs to both
def add_mas(prices):
    p = pd.Series(prices)
    ma5 = p.rolling(5).mean()
    ma20 = p.rolling(20).mean()
    return ma5, ma20

real_ma5, real_ma20 = add_mas(real_prices)
rand_ma5, rand_ma20 = add_mas(random_prices)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Randomize which is on top
import random as rnd
show_real_first = rnd.random() > 0.5

data = [(real_prices, real_ma5, real_ma20, 'Chart A'),
        (random_prices, rand_ma5, rand_ma20, 'Chart B')]
if not show_real_first:
    data = data[::-1]

for ax, (prices, ma5, ma20, title) in zip(axes, data):
    ax.plot(prices, color='white', linewidth=1, label='Price')
    ax.plot(ma5, color='red', linewidth=1.5, label='MA5')
    ax.plot(ma20, color='blue', linewidth=1.5, label='MA20')
    ax.set_title(f'{title} — Which is the real OTC price?', fontsize=14)
    ax.legend()
    ax.set_xlabel('Minutes')

plt.tight_layout()
plt.show()

answer = "A is real" if show_real_first else "B is real"
print(f"\n  Answer: {answer}")
print(f"  Point: Both charts have 'trends' and MAs that seem to predict.")
print(f"  But one is pure random — each step is a coin flip.")
print(f"  The trends you see in the IQ Option chart are the same phenomenon.")

# ═══════════════════════════════════════════════════════════════
# 31: THE REAL QUESTION — What makes OTC different from random?
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("31: What IS Different About OTC vs Random?")
print(f"{'='*60}\n")

# We know some things ARE different:
# 1. Even digits preferred (82% vs 18%)
# 2. Kurtosis = 133 (vs 0 for normal)
# 3. Hourly biases (54-56% at specific hours)
# 4. Price drops at :59→:00 boundary

# Let's quantify: how different IS the OTC data from random?
print("  Comparing OTC data to a random walk:")
print()

real_returns = np.diff(real_prices)
rand_returns = steps

from scipy.stats import ks_2samp, kurtosis, skew

ks_stat, ks_p = ks_2samp(np.diff(trades['close'].values), 
                           np.random.randn(len(trades)) * np.std(np.diff(trades['close'].values)))
print(f"  Distribution test (KS): stat={ks_stat:.4f}, p={ks_p:.4f}")
print(f"  → {'Distributions DIFFER!' if ks_p < 0.01 else 'Similar distributions'}")

print(f"\n  OTC returns: kurtosis={kurtosis(np.diff(trades['close'].values)):.1f}, "
      f"skew={skew(np.diff(trades['close'].values)):.3f}")
print(f"  Normal:      kurtosis=0, skew=0")
print(f"  → OTC has MUCH heavier tails than normal/random")

# The question: can heavy tails be exploited?
# Heavy tails mean rare LARGE moves. But they're unpredictable.
print(f"\n  Can we predict WHEN large moves happen?")
c_big = trades.copy()
c_big['is_big'] = (c_big['move_abs'] > c_big['move_abs'].quantile(0.9)).astype(int)
c_big['prev_is_big'] = c_big['is_big'].shift(1)
c_big = c_big.dropna()

big_after_big = c_big.loc[c_big['prev_is_big'] == 1, 'is_big'].mean()
big_after_small = c_big.loc[c_big['prev_is_big'] == 0, 'is_big'].mean()
print(f"    P(big move | prev was big):   {big_after_big:.1%}")
print(f"    P(big move | prev was small): {big_after_small:.1%}")
print(f"    Base rate:                    {c_big['is_big'].mean():.1%}")

if abs(big_after_big - big_after_small) > 0.02:
    print(f"    → Big moves DO cluster! Potential for volatility prediction")
else:
    print(f"    → No clustering — big moves are random")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 32: THE 1-SECOND DOWN BIAS — Deep Analysis
# ═══════════════════════════════════════════════════════════════
# 53.9% of 1-second moves go DOWN. WHY does this fade at 60s?
# And can we prevent the fade?

print("32: The 1-Second DOWN Bias — Deep Dive")
print("=" * 60)

# ── 32A: Is the bias uniform across all seconds of the minute? ──
print("\n32A: DOWN bias by second-of-minute")
print("-" * 60)

second_bias = []
for sec in range(60):
    sec_data = raw[raw['second'] == sec]['close'].values
    next_sec = raw[raw['second'] == (sec + 1) % 60]['close'].values
    
    # Align: for each second S, find the price 1 second later
    min_len = min(len(sec_data), len(next_sec))
    if sec < 59:
        # Simple: next row in the same minute
        up_pct = (next_sec[:min_len] > sec_data[:min_len]).mean()
    else:
        # :59 → :00 of next minute — use shift
        c59 = raw[raw['second'] == 59]['close']
        c00_next = raw[raw['second'] == 0]['close'].shift(-1)
        # Align by index
        merged = pd.DataFrame({'c59': c59, 'c00': c00_next.reindex(c59.index + pd.Timedelta(seconds=1))})
        merged = merged.dropna()
        if len(merged) > 0:
            up_pct = (merged['c00'].values > merged['c59'].values).mean()
        else:
            up_pct = 0.5
    
    second_bias.append({'second': sec, 'up_pct': up_pct, 'down_pct': 1 - up_pct})

sb_df = pd.DataFrame(second_bias)

fig, ax = plt.subplots(figsize=(16, 5))
colors = ['red' if p < 0.48 else ('lime' if p > 0.52 else 'gray') for p in sb_df['up_pct']]
ax.bar(sb_df['second'], sb_df['up_pct'], color=colors, alpha=0.7)
ax.axhline(y=0.5, color='white', linestyle='--')
ax.axhline(y=0.461, color='yellow', linestyle=':', label='Average (46.1% UP)')
ax.set_title('P(UP) for Each 1-Second Move Within the Minute')
ax.set_xlabel('Second')
ax.set_ylabel('P(UP)')
ax.legend()
ax.set_ylim(0.40, 0.55)
plt.tight_layout()
plt.show()

print(f"  Second-by-second P(UP):")
for _, row in sb_df.iterrows():
    marker = " ★" if row['up_pct'] < 0.44 or row['up_pct'] > 0.52 else ""
    print(f"    :{int(row['second']):02d} → P(UP)={row['up_pct']:.1%}{marker}")

# ── 32B: WHY does bias fade? UP moves must be LARGER than DOWN moves ──
print(f"\n{'='*60}")
print("32B: Move Size Asymmetry — Why Bias Fades")
print("-" * 60)

# 1-second move sizes
one_sec_returns = raw['close'].diff().dropna()
up_returns = one_sec_returns[one_sec_returns > 0]
down_returns = one_sec_returns[one_sec_returns < 0].abs()

print(f"  1-second moves:")
print(f"    UP moves:   count={len(up_returns)}, mean={up_returns.mean()*100000:.3f} pips")
print(f"    DOWN moves: count={len(down_returns)}, mean={down_returns.mean()*100000:.3f} pips")
print(f"    Zero moves: {(one_sec_returns == 0).sum()}")
print(f"    UP/DOWN ratio: {len(up_returns)/len(down_returns):.3f}")
print(f"    Mean UP / Mean DOWN: {up_returns.mean()/down_returns.mean():.3f}")

# Expected drift per second
expected_drift = len(up_returns)/len(one_sec_returns) * up_returns.mean() - len(down_returns)/len(one_sec_returns) * down_returns.mean()
print(f"\n    Expected drift per second: {expected_drift*100000:.4f} pips")
print(f"    Expected drift per minute: {expected_drift*60*100000:.3f} pips")

# So: fewer UP moves but larger → net drift ≈ 0 over 60 seconds
print(f"\n    → Fewer UPs ({len(up_returns)}) but larger ({up_returns.mean()*100000:.3f} pips)")
print(f"    → More DOWNs ({len(down_returns)}) but smaller ({down_returns.mean()*100000:.3f} pips)")
print(f"    → Net effect over 60s: nearly zero drift = 50/50 direction")

# ── 32C: Can we exploit the asymmetry? ──
print(f"\n{'='*60}")
print("32C: Exploiting the Asymmetry")
print("-" * 60)

# The OTC does: many small DOWN steps + few large UP steps = net zero
# What if we can PREDICT which seconds will have large UP steps?
# Those are the seconds that "reset" the DOWN drift.

# Is there a pattern to when large UP moves happen?
one_sec_df = pd.DataFrame({
    'return': one_sec_returns.values,
    'second': raw.index[1:].second,  # which second of the minute
    'is_big_up': (one_sec_returns > one_sec_returns.quantile(0.95)).values,
    'is_big_down': (one_sec_returns < one_sec_returns.quantile(0.05)).values,
})

# Do big UP moves happen at specific seconds?
big_up_by_sec = one_sec_df.groupby('second')['is_big_up'].mean()
big_down_by_sec = one_sec_df.groupby('second')['is_big_down'].mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(big_up_by_sec.index - 0.2, big_up_by_sec.values * 100, width=0.4, 
       color='lime', alpha=0.7, label='Big UP moves')
ax.bar(big_down_by_sec.index + 0.2, big_down_by_sec.values * 100, width=0.4, 
       color='red', alpha=0.7, label='Big DOWN moves')
ax.axhline(y=5, color='white', linestyle=':', label='Expected (5%)')
ax.set_title('When Do Big Moves Happen? (by second of minute)')
ax.set_xlabel('Second')
ax.set_ylabel('% of seconds with big move')
ax.legend()
plt.tight_layout()
plt.show()

# ── 32D: What if we bet DOWN in the first 30s, UP in the last 30s? ──
print(f"\n{'='*60}")
print("32D: Split-Minute Strategy")
print("-" * 60)

# The DOWN drift accumulates in the first half.
# Do large UP corrections happen in the second half?

# Measure: price at :00 vs :30 vs :00(next)
c_split = trades.dropna(subset=['c_30']).copy()
c_split['first_half_up'] = (c_split['c_30'] > c_split['close']).astype(int)
c_split['second_half_up'] = (c_split['result_close'] > c_split['c_30']).astype(int)

first_up = c_split['first_half_up'].mean()
second_up = c_split['second_half_up'].mean()

print(f"  First half  (:00→:30): P(UP) = {first_up:.1%}")
print(f"  Second half (:30→:00): P(UP) = {second_up:.1%}")
print(f"  Full minute (:00→:00): P(UP) = {c_split['went_up'].mean():.1%}")

# If first half is more DOWN and second half is more UP,
# there's a within-minute mean reversion pattern!
if first_up < 0.48 and second_up > 0.52:
    print(f"\n  → WITHIN-MINUTE REVERSAL PATTERN!")
    print(f"  → Price drifts DOWN in first 30s, then corrects UP in last 30s")
    print(f"  → Strategy: enter at :30, bet UP?")
    
    # Test this strategy
    entry_30 = c_split['c_30'].values
    result_00 = c_split['result_close'].values
    trade_up = (result_00 > entry_30).astype(int)
    always_up = np.ones(len(trade_up))
    r = backtest(always_up, trade_up)
    show_result("Enter :30, always UP (30s trade)", r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 33: CORRECTED ANALYSIS — Exclude Zero Moves (Ties)
# ═══════════════════════════════════════════════════════════════
# The "53.9% DOWN" was really "46.1% UP + 7.8% ZERO + 46.1% DOWN"
# IQ Option returns money on ties. Real accuracy is 50/50.
# Let's recheck everything excluding ties.

print("33: Corrected Analysis — Excluding Ties")
print("=" * 60)

# ── 33A: True direction bias by timescale (excluding zero moves) ──
print("\n33A: Direction Bias Excluding Ties")
print("-" * 60)

prices = raw['close'].values

for window_sec in [1, 2, 3, 5, 10, 15, 30, 60]:
    step = window_sec
    ups = 0
    downs = 0
    ties = 0
    for i in range(0, len(prices) - step, step):
        diff = prices[i + step] - prices[i]
        if diff > 0: ups += 1
        elif diff < 0: downs += 1
        else: ties += 1
    
    total = ups + downs + ties
    total_non_tie = ups + downs
    up_pct_with_ties = ups / total if total > 0 else 0
    up_pct_no_ties = ups / total_non_tie if total_non_tie > 0 else 0
    tie_pct = ties / total if total > 0 else 0
    
    label = f"{window_sec}s" if window_sec < 60 else f"{window_sec//60}min"
    print(f"  {label:>5}: UP={up_pct_with_ties:.1%}(with ties)  "
          f"UP={up_pct_no_ties:.1%}(excl ties)  "
          f"Ties={tie_pct:.1%}  "
          f"[{ups} up, {downs} dn, {ties} tie]")

# ── 33B: Follow-last excluding ties ──
print(f"\n{'='*60}")
print("33B: Follow-Last Excluding Ties")
print("-" * 60)

# At 1-minute level: if we exclude minutes where the result is a tie
c_notie = trades.copy()
c_notie = c_notie[c_notie['move'] != 0]  # Exclude exact ties
c_notie['prev_went_up'] = c_notie['went_up'].shift(1)
c_notie = c_notie.dropna()

follow_no_tie = (c_notie['went_up'] == c_notie['prev_went_up']).mean()
print(f"  1-min follow-last (excluding ties): {follow_no_tie:.1%} ({len(c_notie)} trades)")

# ── 33C: IQ Option tie handling ──
print(f"\n{'='*60}")
print("33C: How Often Do Ties Happen at 1-Minute?")
print("-" * 60)

all_moves = trades['move'].values
exact_ties = (all_moves == 0).sum()
near_ties = (np.abs(all_moves) < 0.000005).sum()  # Within half a pip

print(f"  Exact ties (move = 0):     {exact_ties} ({exact_ties/len(all_moves)*100:.2f}%)")
print(f"  Near ties (|move| < 0.5pip): {near_ties} ({near_ties/len(all_moves)*100:.1f}%)")
print(f"  Total minutes: {len(all_moves)}")

# What's the smallest non-zero move?
non_zero = np.abs(all_moves[all_moves != 0])
print(f"\n  Smallest non-zero 1-min move: {non_zero.min()*100000:.1f} pips")
print(f"  Median move: {np.median(non_zero)*100000:.1f} pips")
print(f"  Mean move: {non_zero.mean()*100000:.1f} pips")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 34: BETTING PATTERNS — Minimize Busts at 50% Win Rate
# ═══════════════════════════════════════════════════════════════
# The direction is 50/50. But WHICH bets we win/lose matters.
# Can a specific UP/DOWN pattern reduce consecutive losses?

print("34: Betting Patterns — Minimize Consecutive Losses")
print("=" * 60)

actual_dirs = trades['went_up'].values  # The actual sequence of UP/DOWN

def count_max_streak_and_busts(predictions, actuals):
    """Count max consecutive losses and number of 8+ streaks."""
    max_streak = 0
    current = 0
    busts = 0
    streaks = []
    
    for p, a in zip(predictions, actuals):
        if int(p) == int(a):
            if current > 0:
                streaks.append(current)
            current = 0
        else:
            current += 1
            if current >= 8:
                busts += 1
                current = 0  # Reset after bust
        max_streak = max(max_streak, current)
    if current > 0:
        streaks.append(current)
    
    return {
        'max_streak': max_streak,
        'busts': busts,
        'win_rate': sum(1 for p, a in zip(predictions, actuals) if int(p) == int(a)) / len(actuals) * 100,
        'loss_streaks': streaks,
    }

# Test MANY different betting patterns
patterns = {}

# 1. Always UP
patterns['Always UP'] = np.ones(len(actual_dirs))

# 2. Always DOWN
patterns['Always DOWN'] = np.zeros(len(actual_dirs))

# 3. Alternate: UDUDUDUD
patterns['Alternate UD'] = np.array([i % 2 for i in range(len(actual_dirs))])

# 4. Double alternate: UUDDUURDD
patterns['Double UUDD'] = np.array([(i // 2) % 2 for i in range(len(actual_dirs))])

# 5. Triple: UUUDDDUU
patterns['Triple UUUDDD'] = np.array([(i // 3) % 2 for i in range(len(actual_dirs))])

# 6. Quad: UUUUDDDD
patterns['Quad UUUUDDDD'] = np.array([(i // 4) % 2 for i in range(len(actual_dirs))])

# 7. Five: UUUUUDDDDD
patterns['Five UUUUUDDDDD'] = np.array([(i // 5) % 2 for i in range(len(actual_dirs))])

# 8. Seven: UUUUUUUDDDDDDD
patterns['Seven'] = np.array([(i // 7) % 2 for i in range(len(actual_dirs))])

# 9. Follow-last
patterns['Follow last'] = np.concatenate([[0], actual_dirs[:-1]])

# 10. Fade-last (opposite of previous result)
patterns['Fade last'] = np.concatenate([[0], 1 - actual_dirs[:-1]])

# 11. Random
np.random.seed(42)
patterns['Random'] = np.random.randint(0, 2, size=len(actual_dirs))

# 12. Fibonacci-like: 1,1,2,3,5,8... mod 2
fib_pattern = []
a, b = 0, 1
for _ in range(len(actual_dirs)):
    fib_pattern.append(a % 2)
    a, b = b, a + b
patterns['Fibonacci mod 2'] = np.array(fib_pattern)

# 13. Based on minute index mod various primes
for mod in [3, 5, 7, 11, 13]:
    patterns[f'Minute mod {mod}'] = np.array([((i // mod) % 2) for i in range(len(actual_dirs))])

# Run all patterns
print(f"\n  {'Pattern':<25} {'Win%':>6} {'MaxStreak':>10} {'Busts(8+)':>10}")
print(f"  {'-'*55}")

results_pat = []
for name, pred in patterns.items():
    r = count_max_streak_and_busts(pred, actual_dirs)
    results_pat.append((name, r))

# Sort by busts (fewer = better)
results_pat.sort(key=lambda x: (x[1]['busts'], x[1]['max_streak']))

for name, r in results_pat:
    marker = " ★" if r['busts'] <= 5 else ""
    print(f"  {name:<25} {r['win_rate']:>5.1f}% {r['max_streak']:>10} {r['busts']:>10}{marker}")

# ── Show the loss streak distribution for top patterns ──
print(f"\n{'='*60}")
print("Loss Streak Distributions — Top Patterns")
print(f"{'='*60}\n")

for name, r in results_pat[:5]:
    streaks = r['loss_streaks']
    if streaks:
        print(f"  {name}:")
        print(f"    Max streak: {r['max_streak']}, Busts: {r['busts']}")
        for length in [1, 2, 3, 4, 5, 6, 7, 8]:
            count = sum(1 for s in streaks if s == length)
            if count > 0:
                print(f"      {length} losses in a row: {count} times")
        print()

# ── Full martingale backtest of best patterns ──
print(f"\n{'='*60}")
print("Martingale Backtests — Best Patterns")
print(f"{'='*60}\n")

for name, r_info in results_pat[:8]:
    pred = patterns[name] if name in patterns else None
    if pred is not None:
        r = backtest(pred, actual_dirs)
        show_result(name, r)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 35: SYSTEMATIC PATTERN SEARCH — Find Minimum Busts
# ═══════════════════════════════════════════════════════════════
# UUUDDD had 10 busts. Can we find a pattern with FEWER?
# Search all possible repeating patterns of length 2-12.

print("35: Systematic Pattern Search — Minimize Busts")
print("=" * 60)

from itertools import product

actual = actual_dirs

best_patterns = []

# Test all binary patterns of length 2-10
for pattern_len in range(2, 11):
    best_for_len = None
    best_busts = 999
    best_max = 999
    
    # For short patterns, test ALL possibilities
    if pattern_len <= 8:
        for bits in product([0, 1], repeat=pattern_len):
            # Skip all-same patterns (equivalent to Always UP/DOWN)
            if len(set(bits)) == 1:
                continue
            
            # Generate full prediction sequence by repeating pattern
            pred = np.array([bits[i % pattern_len] for i in range(len(actual))])
            r = count_max_streak_and_busts(pred, actual)
            
            if r['busts'] < best_busts or (r['busts'] == best_busts and r['max_streak'] < best_max):
                best_busts = r['busts']
                best_max = r['max_streak']
                best_for_len = {
                    'pattern': ''.join('U' if b else 'D' for b in bits),
                    'length': pattern_len,
                    'busts': r['busts'],
                    'max_streak': r['max_streak'],
                    'win_rate': r['win_rate'],
                }
    else:
        # For longer patterns, sample random ones
        for _ in range(500):
            bits = tuple(np.random.randint(0, 2, size=pattern_len))
            if len(set(bits)) == 1:
                continue
            pred = np.array([bits[i % pattern_len] for i in range(len(actual))])
            r = count_max_streak_and_busts(pred, actual)
            if r['busts'] < best_busts or (r['busts'] == best_busts and r['max_streak'] < best_max):
                best_busts = r['busts']
                best_max = r['max_streak']
                best_for_len = {
                    'pattern': ''.join('U' if b else 'D' for b in bits),
                    'length': pattern_len,
                    'busts': r['busts'],
                    'max_streak': r['max_streak'],
                    'win_rate': r['win_rate'],
                }
    
    if best_for_len:
        best_patterns.append(best_for_len)

# Sort by busts
best_patterns.sort(key=lambda x: (x['busts'], x['max_streak']))

print(f"\n  {'Pattern':<20} {'Len':>4} {'Win%':>6} {'MaxStrk':>8} {'Busts':>6}")
print(f"  {'-'*48}")
for p in best_patterns:
    marker = " ★" if p['busts'] <= best_patterns[0]['busts'] else ""
    print(f"  {p['pattern']:<20} {p['length']:>4} {p['win_rate']:>5.1f}% {p['max_streak']:>8} {p['busts']:>6}{marker}")

# ── Backtest the absolute best pattern ──
best = best_patterns[0]
print(f"\n{'='*60}")
print(f"BEST PATTERN: {best['pattern']} (period {best['length']})")
print(f"{'='*60}\n")

bits = [1 if c == 'U' else 0 for c in best['pattern']]
pred = np.array([bits[i % len(bits)] for i in range(len(actual))])
r = backtest(pred, actual)
show_result(f"Best pattern: {best['pattern']}", r)

# Compare to random and always-down
r_rand = backtest(np.random.randint(0, 2, size=len(actual)), actual)
show_result("Random baseline", r_rand)

r_down = backtest(np.zeros(len(actual)), actual)
show_result("Always DOWN", r_down)

# ── Is this pattern robust? Test on train/test split ──
print(f"\n{'='*60}")
print("Robustness: Train/Test Split")
print(f"{'='*60}\n")

split = int(len(actual) * 0.5)
train_actual = actual[:split]
test_actual = actual[split:]

# Find best pattern on TRAINING data
best_train = None
best_train_busts = 999
for pattern_len in range(2, 9):
    for bits in product([0, 1], repeat=pattern_len):
        if len(set(bits)) == 1:
            continue
        pred_t = np.array([bits[i % pattern_len] for i in range(len(train_actual))])
        r_t = count_max_streak_and_busts(pred_t, train_actual)
        if r_t['busts'] < best_train_busts:
            best_train_busts = r_t['busts']
            best_train = bits

# Apply to TEST data
if best_train:
    train_pattern = ''.join('U' if b else 'D' for b in best_train)
    pred_test = np.array([best_train[i % len(best_train)] for i in range(len(test_actual))])
    r_test = count_max_streak_and_busts(pred_test, test_actual)
    
    print(f"  Best pattern from training: {train_pattern}")
    print(f"  Training busts: {best_train_busts}")
    print(f"  Test busts:     {r_test['busts']}")
    print(f"  Test max streak: {r_test['max_streak']}")
    print(f"  Test win rate:   {r_test['win_rate']:.1%}")
    
    r_bt = backtest(pred_test, test_actual)
    show_result(f"Best pattern on TEST data", r_bt)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 36: ADAPTIVE BUST-AVOIDANCE STRATEGIES
# ═══════════════════════════════════════════════════════════════
# Fixed patterns don't generalize. But ADAPTIVE rules might.
# The goal: minimize consecutive losses, not predict direction.

print("36: Adaptive Bust-Avoidance Strategies")
print("=" * 60)

actual = actual_dirs

def simulate_adaptive(actual, strategy_fn, label=""):
    """
    Run an adaptive strategy. strategy_fn(state) returns 0 or 1.
    State tracks: current direction, loss streak, trade count, results.
    """
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    wins = 0; losses = 0; busts = 0
    
    state = {'direction': 0, 'loss_streak': 0, 'trade_num': 0, 
             'last_results': [], 'total_wins': 0, 'total_losses': 0}
    
    for a in actual:
        pred = strategy_fn(state)
        
        if consec >= 8 or stake > 200:
            busts += 1; stake = 1.0; consec = 0
        if stake > balance: break
        
        won = int(pred) == int(a)
        if won:
            balance += stake * 0.85
            wins += 1; stake = 1.0; consec = 0
            state['loss_streak'] = 0
            state['total_wins'] += 1
        else:
            balance -= stake
            losses += 1; consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
            state['loss_streak'] += 1
            state['total_losses'] += 1
        
        state['trade_num'] += 1
        state['last_results'].append(int(a))
        if len(state['last_results']) > 20:
            state['last_results'] = state['last_results'][-20:]
        state['direction'] = int(a)  # Track actual direction
        
        history.append(balance)
    
    total = wins + losses
    return {
        'label': label,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'trades': total,
        'busts': busts,
        'profit': balance - 1000,
        'max_dd': 1000 - min(history),
        'history': history,
    }

strategies = []

# S1: Follow last (baseline)
def s_follow(state):
    return state['direction']
r = simulate_adaptive(actual, s_follow, "Follow last")
strategies.append(r)

# S2: Switch direction after N consecutive losses
for switch_after in [2, 3, 4, 5]:
    def make_switch(n):
        def s(state):
            if state['loss_streak'] >= n:
                return 1 - state['direction']  # Flip
            return state['direction']  # Follow
        return s
    r = simulate_adaptive(actual, make_switch(switch_after), f"Switch after {switch_after}L")
    strategies.append(r)

# S3: Alternate after loss (lose → switch, win → keep)
def s_alt_loss(state):
    if state['loss_streak'] > 0:
        return 1 - state['direction']
    return state['direction']
r = simulate_adaptive(actual, s_alt_loss, "Switch on any loss")
strategies.append(r)

# S4: Follow last but flip every N trades regardless
for n in [3, 5, 7, 10]:
    def make_flip(period):
        def s(state):
            if (state['trade_num'] // period) % 2 == 0:
                return state['direction']
            else:
                return 1 - state['direction']
        return s
    r = simulate_adaptive(actual, make_flip(n), f"Follow, flip every {n}")
    strategies.append(r)

# S5: Majority vote of last N results
for n in [3, 5, 7]:
    def make_majority(window):
        def s(state):
            if len(state['last_results']) >= window:
                recent = state['last_results'][-window:]
                return 1 if sum(recent) > window / 2 else 0
            return state['direction']
        return s
    r = simulate_adaptive(actual, make_majority(n), f"Majority of last {n}")
    strategies.append(r)

# S6: Opposite of majority (contrarian)
for n in [3, 5, 7]:
    def make_contrarian(window):
        def s(state):
            if len(state['last_results']) >= window:
                recent = state['last_results'][-window:]
                return 0 if sum(recent) > window / 2 else 1
            return state['direction']
        return s
    r = simulate_adaptive(actual, make_contrarian(n), f"Contrarian last {n}")
    strategies.append(r)

# S7: Random switch — flip direction with probability p after each trade
for p in [0.3, 0.5]:
    def make_random_switch(prob):
        rng = np.random.RandomState(42)
        def s(state):
            if rng.random() < prob:
                return 1 - state['direction']
            return state['direction']
        return s
    r = simulate_adaptive(actual, make_random_switch(p), f"Random flip p={p}")
    strategies.append(r)

# S8: Double-or-nothing: after loss, same direction; after 2 losses, switch
def s_double_switch(state):
    if state['loss_streak'] == 0:
        return state['direction']
    elif state['loss_streak'] % 2 == 0:
        return 1 - state['direction']  # Switch on even losses
    else:
        return state['direction']  # Keep on odd losses
r = simulate_adaptive(actual, s_double_switch, "Switch on even losses")
strategies.append(r)

# S9: Pure alternating (ignore results entirely)
counter = [0]
def s_pure_alt(state):
    counter[0] += 1
    return counter[0] % 2
r = simulate_adaptive(actual, s_pure_alt, "Pure alternate UDUD")
strategies.append(r)

# S10: UUDD pattern (ignore results)
counter2 = [0]
def s_uudd(state):
    counter2[0] += 1
    return (counter2[0] // 2) % 2
r = simulate_adaptive(actual, s_uudd, "Pure UUDD")
strategies.append(r)

# Sort by busts then profit
strategies.sort(key=lambda x: (x['busts'], -x['profit']))

print(f"\n  {'Strategy':<30} {'Win%':>6} {'Busts':>6} {'MaxDD':>8} {'Profit':>9}")
print(f"  {'-'*65}")
for r in strategies:
    bust_str = f"✓ {r['busts']}" if r['busts'] <= 5 else f"  {r['busts']}"
    profit_str = f"✓ +${r['profit']:.0f}" if r['profit'] > 0 else f"  ${r['profit']:.0f}"
    marker = " ★" if r['busts'] <= strategies[0]['busts'] else ""
    print(f"  {r['label']:<30} {r['win_rate']:>5.1f}% {bust_str:>6} ${r['max_dd']:>7.0f} {profit_str:>9}{marker}")

# Equity curves for top strategies
fig, ax = plt.subplots(figsize=(16, 7))
for r in strategies[:6]:
    ax.plot(r['history'], label=f"{r['label']} ({r['busts']} busts)", linewidth=1.5)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Adaptive Strategies — Bust Avoidance')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()